# CRIS Tables

Every prepared table the three query notebooks read, in the order they have to be
built. Nothing here answers a question; run it, then open the section you need.

## Why the builds are not in the section notebooks

They were, and two of them defined the same table. `CRIS_TELECAST_FACT` was
created in both section 1 and section 2 with `CREATE OR REPLACE`, and the two
definitions were not identical: section 2's carried `MEDIAN_AGE`,
`ORIGINATOR_ID`, `MEDIA_TYPE_CODE`, `COMMERCIAL_DURATION` and the skew columns,
and section 1's did not.

So whichever notebook ran last won. Running section 1 after section 2 silently
dropped five columns, and 2.5, 2.6 and the section 3 rankers then failed on an
invalid identifier, with an error pointing at the query rather than at the
notebook that had overwritten the table underneath it.

One table, one definition, one place to run it.

## Build order

The order matters twice over, once for dependencies and once for currency.

| step | table | serves | depends on |
|---|---|---|---|
| 1 | `CRIS_TELECAST_FACT` | 1.1, 1.3, 1.3b, 1.5, most of section 2 | the unified total-program view |
| 2 | `CRIS_HALFHR_FACT` | 2.9, 2.13, 3.1, 3.2, 3.3, and 3.4's daypart test | the unified half-hour view |
| 3 | `CRIS_QH_FACT` | 1.2, 1.4, 2.11 | the unified quarter-hour view |
| 4 | `CRIS_LATEST_DATE` | every "last night" default | steps 1 and 3 |
| 5 | `CRIS_NETWORK_UNIVERSE` | 2.9 and all of section 3 | step 2 |
| 6 | `CRIS_FY_CALENDAR` | 3.2, 3.3 | A&E's own `Dim_DATE` |
| 7 | `CRIS_TELECAST_ALL_NET` | 3.4 | the unified total-program view |

Steps 1 to 3 are independent of each other and can run in any order among
themselves. Step 4 reads the two before it, so it has to follow them.

## Rebuild all of them, or none

Partial rebuilds are the failure mode worth naming, because they do not fail.
3.4 takes its date window from `CRIS_TELECAST_ALL_NET` and decides daypart
membership from `CRIS_HALFHR_FACT`. When the second is two weeks behind the
first, the join finds nothing, the ranker returns an empty table, and an empty
ranker reads exactly like a quiet week. No error appears anywhere.

The last cell of this notebook prints how far each table reaches against its
source view. Run it after every rebuild. Anything sitting below its source needs
building again.

**Version 70.** All CRIS notebooks are renumbered to 50 together; see the note below.

> **v70.** No change to the tables from v59. Renumbered with the set.


> **v59.** ⚠ **`IS_PREMIERE_CT` is now the season premiere**, computed here: the first first-run telecast of each season, by broadcast date and telecast number. Cable Tracks' own column is kept beside it as **`IS_PREMIERE_CT_SOURCE`**.
>
> **No query changes.** Everything that already reads `IS_PREMIERE_CT` keeps working and now gets the right value: 1 on the season opener only, 0 elsewhere. The type stays 1/0, so `= 1` comparisons are unaffected.
>
> **Why.** On HIST GREATEST MACHINES season 1 the source column reads 1 on all eight episodes, so it marks first runs rather than the season opener. That is why "how did the season premiere perform" came back with the season average of 221 over 8 telecasts instead of AERIAL ATTACKERS at 266. Adarsh was going to confirm the column's meaning with the data team and that is still open; keeping the source column means the answer can be compared once it arrives.
>
> **Rebuild the three fact tables.** `CRIS_LATEST_DATE` and `CRIS_NETWORK_UNIVERSE` do not read either column, so they only need rebuilding if the data itself has moved.


> **v58.** No change from v57. Every query validated running against the rebuilt tables: 22 of 22 execute without error across the three sections. Renumbered so the whole set carries one version.


> **v57.** No change to the tables from v56. Renumbered with the query notebooks and documents.


> **v55.** ⚠ **Two Cable Tracks columns, one per meaning.** Agreed in the tech biweekly, 18 Sep 2026.
>
> | column | level | source |
> |---|---|---|
> | `IS_FIRST_RUN_CT` | episode: first run or repeat | derived, `CT_SEASON IS NOT NULL` |
> | `IS_PREMIERE_CT` | season: first episode of the season | Cable Tracks, from the view |
>
> v54 named the derived column `IS_PREMIERE_CT`, which collided with a real column in the source view that nobody had found because it carries the `_CT` suffix rather than the `CT_` prefix. The derived one is renamed and the view's is carried through unchanged.
>
> Nielsen's `REPEAT_IND` and `IS_SEASON_PREMIERE` stay on the tables so the sources can be compared. No query reads them.


> **v54.** ⚠ **`CRIS_TELECAST_ALL_NET` is removed.** With the six-network filter gone from `CRIS_TELECAST_FACT`, the two held identical rows. 3.4 was the only query naming it and now reads the telecast fact.
>
> **Six tables, not seven:** `CRIS_TELECAST_FACT`, `CRIS_HALFHR_FACT`, `CRIS_QH_FACT`, `CRIS_LATEST_DATE`, `CRIS_NETWORK_UNIVERSE`, `CRIS_FY_CALENDAR`.
>
> Once the rebuild is verified, drop the old one by hand:
> `DROP TABLE AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_ALL_NET;`


> **v53.** ⚠ **Every network, in every fact table** `[ALLNETS]`. The six-network filter is removed from `CRIS_TELECAST_FACT` and `CRIS_QH_FACT`. `CRIS_HALFHR_FACT` and `CRIS_TELECAST_ALL_NET` already held every network, so all four now match.
>
> The filter was there because Sections 1 and 2 only ever asked about A&E networks. What it cost is that a question about any other network returned zero rows and no error, which reads as a quiet night rather than as out of scope.
>
> **This makes the tables roughly twenty-five times larger.** Rebuild time and storage both go up accordingly.
>
> **No validated figure moves.** Every query filters on `p_network`, and the six behave exactly as before. What changes is that a seventh now returns rows instead of nothing.
>
> **`CRIS_TELECAST_ALL_NET` is now a duplicate of `CRIS_TELECAST_FACT`.** It is kept because the Section 3 queries name it, and folding it in means editing those. Worth doing, separately.
>
> ⚠ **`IS_PREMIERE_CT` added to all four fact tables** `[CT-PREMIERE]`, carried over from v52. Cable Tracks presence as the premiere indicator, signed off by Jill on 11 Sep 2026. Nothing reads it yet and `REPEAT_IND` is untouched, so the two can be compared before anything is migrated.
>
> **Rebuild order:** `CRIS_TELECAST_FACT`, `CRIS_HALFHR_FACT`, `CRIS_QH_FACT`, `CRIS_TELECAST_ALL_NET`, then `CRIS_LATEST_DATE` and `CRIS_NETWORK_UNIVERSE` last, since both read from the tables above.


> **v52.** ⚠ **`IS_PREMIERE_CT` added to all four fact tables** `[CT-PREMIERE]`: `CRIS_TELECAST_FACT`, `CRIS_HALFHR_FACT`, `CRIS_QH_FACT` and `CRIS_TELECAST_ALL_NET`.
>
> Cable Tracks presence as the premiere indicator, signed off by Jill in the working session of 11 Sep 2026 and logged here at Adarsh's request so the decision carries a date: *a telecast Cable Tracks has catalogued is a premiere, one it has not is a repeat, and Nielsen's flag is not used for this.*
>
> **Nothing reads it yet.** `REPEAT_IND` is untouched and no query has been migrated, so every validated figure holds. The two columns sit side by side on purpose: their disagreement is the measurement Adarsh asked for. On `THE FIRST 48` they differ on 4,408 of 4,892 Nielsen premieres, and he confirmed in the session that those are late-night mirror airings, with no missing seasons at all in the 8p-12a daypart.
>
> Migrating the queries is a separate step and it moves numbers: `THE FIRST 48` goes from 4,892 premieres to 484. The check cell at the end of this notebook measures the disagreement per network before anything is switched over.
>
> **Not the same thing as `IS_SEASON_PREMIERE`**, which is the first episode of a season. That one is still blocked on Adarsh confirming whether a Cable Tracks premiere flag exists.
>
> **Rebuild all four tables.** `CRIS_NETWORK_UNIVERSE` reads `CRIS_HALFHR_FACT`, so rebuild it afterwards to keep the two in step.


> **v51.** ⚠ **`IS_HOLIDAY` added to `CRIS_HALFHR_FACT` and `CRIS_TELECAST_ALL_NET`** `[S2-HOL]`, computed identically to `CRIS_TELECAST_FACT`: `MOVIE_IND = 1` and `PROGRAM_NAME` matching `'% HOL %'` or `'%HOLIDAY%'`.
>
> It was only ever on the telecast table, which is why a holiday question could be asked about a telecast but not about a daypart or a competitor. Nothing blocked it - the rule is derived from `PROGRAM_NAME`, and all three tables already carry `PROGRAM_NAME` and `MOVIE_IND`. It simply was never carried across.
>
> **This is additive.** Existing columns are untouched and no validated figure moves. Both tables must be rebuilt for the column to appear; `CRIS_NETWORK_UNIVERSE` reads `CRIS_HALFHR_FACT`, so rebuild it afterwards to keep the two in step.
>
> **Blocks SP_5.** *"Which holiday movie on LFT had the highest L+3 lift in FY 2025 among P25-64?"* needs `p_holiday = TRUE` in 2.7, and the parameter has nothing to filter on without this column.
>
> **Known limit, unchanged:** only films can be `TRUE`. A holiday special that is not a movie is invisible to this flag on all three tables. Carried in Still open rather than fixed, because widening it means deciding what counts as holiday content without a Nielsen field to appeal to.

> **v50.** Version reset. Every CRIS notebook restarts at 50 from here, replacing the drift that had Tables on 18, Section 3 on 23, Section 2 on 4, Section 1 on 1 and the resolver on 2. The convention is unchanged - all notebooks carry one number and move together - but the numbers themselves had stopped agreeing, and there was no way to tell from a filename whether a notebook was current or five versions behind. 50 is far enough above every number in circulation that no old file can be mistaken for a new one.
>
> ⚠ **`HAS_CT_COVERAGE` and `CT_SEASON_PCT` added** `[S3-CTCOV]`. Cable Tracks covers eight networks, not six: HGTV and BRVO come back at 97.4% against A&E's 98.1%, and the next network down is FOX at 26.7%. Measured Aug-20-2026 against `VW_TOTAL_PROGRAM_ESTIMATES_UNIFIED` directly, so it is a source limit rather than something this build dropped. The resolver gates season, genre and specials questions on this flag instead of on `IS_AE`, which was wrong for HGTV and Bravo.
>
> **No network name column exists** `[S3-NAME]`. Checked every source view Aug-20-2026: there is no display name anywhere, only `NETWORK_CODE`. Name-to-code resolution has to live in the planner. `NETWORK_OWNER` is on the views but is the corporate parent, not the channel, and `CRIS_HALFHR_FACT` does not carry it, so it is not added here.
>
> **Only `CRIS_NETWORK_UNIVERSE` changes.** The other six tables are byte-identical to v18 and do not need rebuilding. No output of any existing query changes: `IS_AE`, `IS_CABLE`, `IS_BROADCAST`, `IS_ARTIFACT`, `IS_ENT_BY_CODE` and `IS_ENT_BY_NAME` are computed exactly as before, so every validated Section 3 figure holds.
>
> **Carried forward from v18 and earlier**, unchanged: the seven tables, their build order, and the three uniqueness checks. The version notes below are kept as history.

> **v18.** All seven notebooks renumbered together.

## Changes v17 -> v18

**A defect in v17's table build, and it is mine.**

`CT_GENRE_2`, `CT_GENRE_3`, `CT_COMBINED_GENRE` and `PROG_TYPE_ALPHA` were added to
the `src` CTE of all four fact tables, which is where they are read from the view -
but the outer `SELECT` that actually defines the table's columns was only updated on
`CRIS_HALFHR_FACT` and `CRIS_TELECAST_ALL_NET`.

On `CRIS_TELECAST_FACT` and `CRIS_QH_FACT` that projection writes `CT_GENRE_1` in a
different neighbourhood - beside `IS_SPECIAL` rather than beside `CT_SEASON` - so the
edit missed it. The columns were pulled from the view and then dropped on the way out.

**Both tables built without an error and without the columns.** That is the failure
mode this notebook already warns about in another form: nothing fails, the answer is
just quietly different. Section 3's 3.3 worked because it reads half hours; anything
reading the telecast table did not have the columns at all.

Rebuild `CRIS_TELECAST_FACT` and `CRIS_QH_FACT` from this version.
`CRIS_HALFHR_FACT` and `CRIS_TELECAST_ALL_NET` are unchanged and do not need it.

### Check it landed

Run this after the rebuild. It should return four rows for every fact table, not
just two of them.

```sql
SELECT TABLE_NAME, COLUMN_NAME
FROM AUDIENCE_DEV_DB.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'CRIS'
  AND COLUMN_NAME IN ('CT_GENRE_1','CT_GENRE_2','CT_COMBINED_GENRE','PROG_TYPE_ALPHA')
ORDER BY TABLE_NAME, COLUMN_NAME;
```

Then `[S2-SYNC]` at the end of this notebook, to confirm all four reach the same date.

> **v17.** All seven notebooks renumbered together.

## Changes v16 -> v17

**⚠ This is a table rebuild.** Four columns added to every fact table:
`CT_GENRE_2`, `CT_GENRE_3`, `CT_COMBINED_GENRE` and `PROG_TYPE_ALPHA`. Additive -
no existing query breaks - but nothing that needs them works until the tables are
rebuilt, and `CRIS_HALFHR_FACT` is three years of half hours across 174 networks.

Rebuild all four, or none. `[S2-SYNC]` at the end of this notebook prints how far
each reaches.

### `[S3-GENRE2]` why

R_1 was returning WWE SMACKDOWN at #3 in a ranker it does not appear in at all, and
`CT_GENRE_1` cannot explain why:

| programme | CT_GENRE_1 | CT_GENRE_2 |
|---|---|---|
| WWE SMACKDOWN (USA) | NON-FICTION | **SPORTS** |
| WWE NXT (CW) | NON-FICTION | **SPORTS** |
| TOURNAMENT OF CHAMPIONS | NON-FICTION | REALITY COMPETITION |
| CURSE OF OAK ISLAND | NON-FICTION | NON-FICTION ADVENTUR |
| 1000-LB SISTERS | NON-FICTION | REALITY DOCU-DRAMA |

Everything in A&E's ranker carries a reality or non-fiction second genre; everything
that should not be there carries SPORTS. Their own sheet prints a Genre 2 column with
exactly those values.

The build stored only `CT_GENRE_1`. The view carries `CT_GENRE_1` through
`CT_GENRE_4` plus `CT_COMBINED_GENRE`, so the data was always there and the table was
throwing it away.

### `PROG_TYPE_ALPHA` was a broken promise

3.3's `p_genre_col` offers `'PROG_TYPE_ALPHA'` as the fallback for when Cable Tracks
is thin on competitors. That column lived only on `CRIS_TELECAST_ALL_NET`, and 3.3
reads `CRIS_HALFHR_FACT`, so choosing it failed on an invalid identifier. Nobody had
run it.

The fallback is not hypothetical: in the R_1 window every `WWE RIVALS` row on COM and
every ESPN row comes back with `CT_GENRE_1` NULL. Cable Tracks genuinely does not
cover parts of the competitive landscape, which is the case it was written for.

`CRIS_TELECAST_ALL_NET` already carried it and is not duplicated.

### What this does not fix

A&E's R_1 header also reads *All Exclusions & **Mini-Series***, and genre does not
separate those: `90 DAY FIANCE TOW: TELL A` and `90 DAY FIANCE BT90D: TELL` both
carry `REALITY REAL LIFE`, the same as the parent series, over two telecasts each.
They are distinct `PROGRAM_NAME`s that A&E leave out and we keep. Open with A&E:
whether Mini-Series means dropping the short companion runs or consolidating them
under the parent.

> **v16.** All seven notebooks renumbered together. **Only
> `CRIS_Section3_Queries_v16` and `CRIS_Benchmark_Section3_v16` changed**, with
> three parameters from the first Section 3 run: `[S3-PREMIERES]`,
> `[S3-SLOTMIN]` and `[S3-FYTO]`.
>
> This file is byte-identical to its v15.

> **v16.** All seven notebooks renumbered together, and there are now seven:
> `CRIS_Benchmark_Section3_v16` is new. **No query changed** - the new file is a
> benchmark, built from `CRIS_Section3_Queries_v16` with parameters only.
>
> This file is byte-identical to its v14.

> **v16.** All six notebooks renumbered together. **Only
> `CRIS_Section2_Queries_v16` and `CRIS_Benchmark_Section2_v16` changed**, with
> `[S2-NORMCLIP]`: 2.2's premiere norm now reads `CRIS_HALFHR_FACT` so the daypart
> can be clipped to the part of each telecast inside prime. That reproduces A&E's
> 250 over 142 nights and 17,155 minutes.
>
> This file is byte-identical to its v13.

> **v15.** All six notebooks renumbered together. **Only
> `CRIS_Benchmark_Section1_v15` changed**: its three cells carried a stale pasted
> copy of 1.1 and are regenerated from `CRIS_Section1_Queries_v15`. No calculated
> figure moves.
>
> This file is byte-identical to its v12.

> **v14.** All six notebooks renumbered together. **Only
> `CRIS_Section2_Queries_v14` and `CRIS_Benchmark_Section2_v14` changed**, with
> `[S2-NORMWIN]`: the norm on 2.1, 2.2, 2.5 and 2.6 can now take its own window,
> separate from the series'. Both new parameters default to NULL, which means
> *same as the series window*, so no validated figure moves.
>
> This file is byte-identical to its v11.

> **v13.** Renumbered from v10 so the six notebooks stay on one number.
> **Only Section 1 changed**, and only in display: `PRIOR_SEASON`,
> `PRIOR_PREMIERE_K`, `PRIOR_PREMIERE_TC` and `CHANGE_VS_PRIOR_PREMIERE_PCT` now
> print only where `NORM_BASIS` is a prior season. They printed on every row,
> which produced a real misreading in the v10 run - RACHAEL RAYS' prior-season
> premiere also averaged 8, the same as its season norm, so two different columns
> both read +300.0 and looked like a defect. No calculated figure changed.
>
> Everything else is byte-identical to v10: `CRIS_Tables_v13`,
> `CRIS_Section2_Queries_v13`, `CRIS_Section3_Queries_v13`,
> `CRIS_Benchmark_Section1_v13`, `CRIS_Benchmark_Section2_v13`.
>
> The v10 run stands: OV_1, OV_2, OV_3, SP_1 through SP_6, SP_8 and SP_9 all
> reproduce exactly, and SP_7 is short two figures for the reason set out in
> `CRIS_Benchmark_Section2_v13`.

> **v12.** All six CRIS notebooks carry the same version number from here on -
> `CRIS_Tables_v12`, `CRIS_Section1_Queries_v12`, `CRIS_Section2_Queries_v12`,
> `CRIS_Section3_Queries_v12`, `CRIS_Benchmark_Section1_v12`,
> `CRIS_Benchmark_Section2_v12`. They were on v4, v5, v6, v7 and v1 at once,
> which is how `CRIS_TELECAST_FACT` came to be created in two places with two
> different column lists. One number, one set.

## v5 -> v12

**Nothing changed** in the seven prepared tables. Renumbered only, so the six notebooks share a version.

Section 3 has still never been compared against A&E's R_1 to R_6, which have full
pulls in the answer workbook. That is the next benchmark to write.

## 1. `CRIS_TELECAST_FACT`


In [ ]:
%%sql -r dataframe_1
-- ============================================================================
-- [S2-T0] CRIS_TELECAST_FACT
-- ----------------------------------------------------------------------------
-- THE ONLY DEFINITION OF THIS TABLE. Section 1 reads it and does not build it.
--
-- It used to be built in both notebooks with CREATE OR REPLACE, and the two
-- definitions were not the same: this one carries MEDIAN_AGE, ORIGINATOR_ID,
-- MEDIA_TYPE_CODE, COMMERCIAL_DURATION and the skew columns, and [S1-T1] did
-- not. Whichever notebook ran last therefore won. Running section 1 afterwards
-- silently dropped five columns and 2.5, 2.6 and the rankers stopped compiling,
-- with an error that points at the query rather than at the notebook that
-- overwrote the table underneath it.
--
-- One table, one definition. The extra columns cost nothing to a section 1
-- query, which simply does not select them.
--
--     MEDIAN_AGE           2.5
--     P2 / M2 / F2         2.6, and the denominator for any skew
--     COMMERCIAL_DURATION  ACM weighting, C3 and C7 in section 3
--     ORIGINATOR_ID        the competitive universe in section 3
--
-- Same grain, same source, same filters, same [S1-FEED] / [S1-VT] / [S1-DEDUP]
-- ordering that section 1 documents in full. Additive by construction, so every
-- section 1 number is unchanged.
--
-- Build order: 1 of 3. Run this before anything in section 1 or 3.
-- ============================================================================
-- [ALLNETS] CRIS_TELECAST_FACT now also holds every network, so this table and
-- that one carry the same rows. It is kept because Section 3 queries name it and
-- retiring it would mean editing them; it should be folded into the telecast
-- fact once those are pointed at the one table.
CREATE OR REPLACE TABLE AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
CLUSTER BY (BROADCAST_DATE, NETWORK_CODE)
AS
WITH src AS (
    SELECT
        NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE, DAY_OF_WEEK,
        PROGRAM_CODE, TELECAST_NUM, EPISODE_ID,
        PROGRAM_NAME, EPISODE_NAME, CT_SEASON, IS_PREMIERE_CT,
        -- [S3-GENRE2] four genre columns, not one. CT_GENRE_1 alone cannot
        -- separate WWE SMACKDOWN from TOURNAMENT OF CHAMPIONS: both read
        -- NON-FICTION, and it is CT_GENRE_2 that says SPORTS against
        -- REALITY COMPETITION. A&E's R_1 ranker prints a Genre 2 column and
        -- excludes the sports rows; ours could not, so WWE SMACKDOWN came
        -- back at #3 in a ranker it does not appear in at all.
        --
        -- CT_COMBINED_GENRE is the concatenation A&E display, carried so the
        -- answer layer does not rebuild it.
        --
        -- PROG_TYPE_ALPHA is Nielsen's own taxonomy and is here because
        -- 3.3's p_genre_col offers it as the fallback for when Cable Tracks
        -- is thin on competitors - and that option was BROKEN, because the
        -- column existed only on CRIS_TELECAST_FACT. It is not
        -- hypothetical either: every WWE RIVALS row on COM and every ESPN
        -- row in the R_1 window carries CT_GENRE_1 NULL, which is exactly
        -- the case the fallback was written for.
        CT_GENRE_1, CT_GENRE_2, CT_GENRE_3, CT_COMBINED_GENRE,
        PROG_TYPE_ALPHA,
        START_TIME, END_TIME, EVENT_DURATION, CONTRIBUTING_DURATION,
        COMMERCIAL_DURATION,
        EVENT_START_HR, EVENT_START_MIN,
        REPEAT_IND, MOVIE_IND, IS_SPECIAL, SHORT_DURATION_IND,
        PROG_PREMIERE_IND, PROG_ORIG_ACQUIRED_CD, EPISODE_ORIG_ACQUIRED_CD,
        MEDIAN_AGE,
    -- [S2-DEMO] P2 / M2 / F2 are not repeated here: the full AA block below
    -- already carries them, and 2.5 and 2.6 read them from there.
        -- ---- demos: every AA column the source view carries  [S2-DEMO] --------
    -- 74 columns, which is all of them. The table used to keep three, the three
    -- DEMO_AA could resolve to, which made the table the thing deciding which
    -- demo an answer came back in.
    --
    -- Nothing is composed or summed. These are the view's own columns under the
    -- view's own names, so a figure in an answer traces back to a column in the
    -- DDL with no translation step in between.
    --
    -- DEMO_AA stays as the default. It resolves per network, which is what lets
    -- one run cover several networks with each answering in its own demo.
    -- p_demo overrides it per query.
    F12_14_AA_ESTIMATES, F15_17_AA_ESTIMATES, F18_20_AA_ESTIMATES,
    F18_34_AA_ESTIMATES, F18_49_AA_ESTIMATES, F18_AA_ESTIMATES,
    F21_24_AA_ESTIMATES, F21_AA_ESTIMATES, F25_29_AA_ESTIMATES,
    F25_54_AA_ESTIMATES, F25_64_AA_ESTIMATES, F25_AA_ESTIMATES,
    F2_17_AA_ESTIMATES, F2_5_AA_ESTIMATES, F2_AA_ESTIMATES,
    F30_34_AA_ESTIMATES, F35_39_AA_ESTIMATES, F35_49_AA_ESTIMATES,
    F35_54_AA_ESTIMATES, F40_44_AA_ESTIMATES, F45_49_AA_ESTIMATES,
    F50PLUS_AA_ESTIMATES, F50_54_AA_ESTIMATES, F50_64_AA_ESTIMATES,
    F55PLUS_AA_ESTIMATES, F55_64_AA_ESTIMATES, F65_AA_ESTIMATES,
    F6_8_AA_ESTIMATES, F9_11_AA_ESTIMATES, HH_AA_ESTIMATES,
    M12_14_AA_ESTIMATES, M15_17_AA_ESTIMATES, M18_20_AA_ESTIMATES,
    M18_34_AA_ESTIMATES, M18_49_AA_ESTIMATES, M18_AA_ESTIMATES,
    M21_24_AA_ESTIMATES, M21_AA_ESTIMATES, M25_29_AA_ESTIMATES,
    M25_54_AA_ESTIMATES, M25_64_AA_ESTIMATES, M25_AA_ESTIMATES,
    M2_17_AA_ESTIMATES, M2_5_AA_ESTIMATES, M2_AA_ESTIMATES,
    M30_34_AA_ESTIMATES, M35_39_AA_ESTIMATES, M35_49_AA_ESTIMATES,
    M35_54_AA_ESTIMATES, M40_44_AA_ESTIMATES, M45_49_AA_ESTIMATES,
    M50PLUS_AA_ESTIMATES, M50_54_AA_ESTIMATES, M50_64_AA_ESTIMATES,
    M55PLUS_AA_ESTIMATES, M55_64_AA_ESTIMATES, M65_AA_ESTIMATES,
    M6_8_AA_ESTIMATES, M9_11_AA_ESTIMATES, P18_34_AA_ESTIMATES,
    P18_49_AA_ESTIMATES, P18_AA_ESTIMATES, P21_AA_ESTIMATES,
    P25_54_AA_ESTIMATES, P25_64_AA_ESTIMATES, P25_AA_ESTIMATES,
    P2_17_AA_ESTIMATES, P2_AA_ESTIMATES, P35_49_AA_ESTIMATES,
    P35_54_AA_ESTIMATES, P50_64_AA_ESTIMATES, P50_AA_ESTIMATES,
    P55_AA_ESTIMATES, P65_AA_ESTIMATES,
        MEDIA_TYPE_CODE, ORIGINATOR_ID,
        TRACKAGE_ID, FEED_PATTERN_IND, MARKET_BREAK_ID,
        FILE_NAME, FILE_DATE,
        IFF(UPPER(COALESCE(FILE_NAME,'')) LIKE 'UBD%','BIGDATA','PANEL')  AS SOURCE_FEED,
        NULLIF(TRIM(COALESCE(VIEWING_TYPE,'')),'')                        AS VT
    FROM AUDIENCE_DB.UBD.VW_TOTAL_PROGRAM_ESTIMATES_UNIFIED
    -- [ALLNETS] every network, not the six. The filter was here because
    -- Sections 1 and 2 only ever asked about A&E networks, but a question about
    -- any other network then returned zero rows and no error, which reads as a
    -- quiet night rather than as out of scope. CRIS_HALFHR_FACT and
    -- CRIS_TELECAST_FACT already carried every network; these now match.
    WHERE PROGRAM_PUT_INDICATOR = 0
      AND TOTAL_PROGRAM_IND     = '00'
),
ranked AS (
    SELECT src.*,
        IFF(SOURCE_FEED = 'BIGDATA', 0, 1)                                AS FEED_RANK,
        -- [S1-DUP] weekly is the settled delivery, daily the preliminary
        IFF(REGEXP_LIKE(UPPER(COALESCE(FILE_NAME,'')), '.*_CW.*'), 0, 1)  AS WKLY_RANK,
        IFF(VT IN ('1','3','5','B'), 1, 0)                                AS VT_RANK,
        IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(TRACKAGE_ID)),0) = 0, 0, 1) AS TRK_RANK,
        IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(MARKET_BREAK_ID)),0) = 0, 0, 1) AS MKT_RANK
    FROM src
)
SELECT
    NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE, DAY_OF_WEEK,
    PROGRAM_CODE, TELECAST_NUM, EPISODE_ID,
    PROGRAM_NAME, EPISODE_NAME, CT_SEASON,
    START_TIME, END_TIME, EVENT_DURATION, CONTRIBUTING_DURATION,
    COMMERCIAL_DURATION,
    (MOD(MOD(EVENT_START_HR, 24) + 18, 24) + 6) * 100 + EVENT_START_MIN  AS BCAST_START_2959,
    -- [CT-FIRSTRUN] episode level: catalogued in Cable Tracks = first run.
    -- Jill, 11 Sep 2026. IS_PREMIERE_CT, from the view, is the season premiere.
    IFF(CT_SEASON IS NOT NULL, TRUE, FALSE)                         AS IS_FIRST_RUN_CT,
    IS_PREMIERE_CT                                                  AS IS_PREMIERE_CT_SOURCE,
    -- [CT-SEASONPREM] IS_PREMIERE_CT is now OURS: the season premiere, the first
    -- first-run telecast of a season. Cable Tracks' own column is kept beside it
    -- as IS_PREMIERE_CT_SOURCE.
    --
    -- Measured, HIST GREATEST MACHINES season 1: the source column reads 1 on
    -- all eight episodes, so it marks first runs rather than the season opener
    -- and cannot answer "how did the season premiere do". Adarsh was going to
    -- confirm its meaning with the data team; that is still open.
    --
    -- The comparison is against a MIN over the season rather than ROW_NUMBER
    -- because QUALIFY below runs after this: duplicate copies of one telecast
    -- share the same date and telecast number, so whichever copy survives the
    -- dedup still matches the minimum. A ROW_NUMBER could land on a copy that
    -- QUALIFY then drops, leaving a season with no premiere at all.
    --
    -- Repeats carry no CT_SEASON, so they fall in a different partition and can
    -- never be picked.
    IFF(CT_SEASON IS NOT NULL
        AND TO_VARCHAR(BROADCAST_DATE, 'YYYYMMDD')
            || LPAD(TO_VARCHAR(TELECAST_NUM), 12, '0')
          = MIN(TO_VARCHAR(BROADCAST_DATE, 'YYYYMMDD')
                || LPAD(TO_VARCHAR(TELECAST_NUM), 12, '0'))
            OVER (PARTITION BY NETWORK_CODE, RATING_SOURCE,
                               PROGRAM_CODE, CT_SEASON),
        1, 0)                                                       AS IS_PREMIERE_CT,
    REPEAT_IND, MOVIE_IND, IS_SPECIAL, SHORT_DURATION_IND,
    -- [S3-GENRE2] see the src CTE above
    CT_GENRE_1, CT_GENRE_2, CT_GENRE_3, CT_COMBINED_GENRE, PROG_TYPE_ALPHA,
    IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(PROG_PREMIERE_IND)),0) = 1, 1, 0) AS IS_SEASON_PREMIERE,
    CASE WHEN MOVIE_IND = 1 THEN 'Movie'
         WHEN UPPER(COALESCE(CT_GENRE_1,'')) = 'SPECIAL' OR IS_SPECIAL = 1 THEN 'Special'
         ELSE 'Series' END                                               AS CONTENT_TYPE,
    CASE WHEN MOVIE_IND = 1
           AND ( UPPER(PROGRAM_NAME) LIKE '% HOL %'
              OR UPPER(PROGRAM_NAME) LIKE '%HOLIDAY%' )
         THEN TRUE ELSE FALSE END                                        AS IS_HOLIDAY,
    COALESCE(NULLIF(TRIM(COALESCE(EPISODE_ORIG_ACQUIRED_CD,'')),''),
             NULLIF(TRIM(COALESCE(PROG_ORIG_ACQUIRED_CD,'')),''))        AS ORIG_ACQUIRED_CD,
    CASE WHEN UPPER(PROGRAM_NAME) LIKE '%ORIGINAL%' THEN 'ORIGINAL'
         WHEN UPPER(PROGRAM_NAME) LIKE '%ACQUIRED%' THEN 'ACQUIRED'
         WHEN UPPER(PROGRAM_NAME) LIKE '%FEATURE%'  THEN 'FEATURE'
         ELSE NULL END                                                   AS MOVIE_TYPE,
    -- ---- demos ----------------------------------------------------------
    MEDIAN_AGE,                                        -- [S2-DEMO] new, 2.5
    -- [S2-DEMO] every AA column the source view carries, all 74. The src CTE
    -- above already selects them; this is the projection that actually
    -- materialises them into the table, and it previously carried six. A demo
    -- named in a query resolves against THIS list, so anything missing here is
    -- an invalid identifier at query time no matter what the CTE holds.
    F12_14_AA_ESTIMATES, F15_17_AA_ESTIMATES, F18_20_AA_ESTIMATES,
    F18_34_AA_ESTIMATES, F18_49_AA_ESTIMATES, F18_AA_ESTIMATES,
    F21_24_AA_ESTIMATES, F21_AA_ESTIMATES, F25_29_AA_ESTIMATES,
    F25_54_AA_ESTIMATES, F25_64_AA_ESTIMATES, F25_AA_ESTIMATES,
    F2_17_AA_ESTIMATES, F2_5_AA_ESTIMATES, F2_AA_ESTIMATES,
    F30_34_AA_ESTIMATES, F35_39_AA_ESTIMATES, F35_49_AA_ESTIMATES,
    F35_54_AA_ESTIMATES, F40_44_AA_ESTIMATES, F45_49_AA_ESTIMATES,
    F50PLUS_AA_ESTIMATES, F50_54_AA_ESTIMATES, F50_64_AA_ESTIMATES,
    F55PLUS_AA_ESTIMATES, F55_64_AA_ESTIMATES, F65_AA_ESTIMATES,
    F6_8_AA_ESTIMATES, F9_11_AA_ESTIMATES, HH_AA_ESTIMATES,
    M12_14_AA_ESTIMATES, M15_17_AA_ESTIMATES, M18_20_AA_ESTIMATES,
    M18_34_AA_ESTIMATES, M18_49_AA_ESTIMATES, M18_AA_ESTIMATES,
    M21_24_AA_ESTIMATES, M21_AA_ESTIMATES, M25_29_AA_ESTIMATES,
    M25_54_AA_ESTIMATES, M25_64_AA_ESTIMATES, M25_AA_ESTIMATES,
    M2_17_AA_ESTIMATES, M2_5_AA_ESTIMATES, M2_AA_ESTIMATES,
    M30_34_AA_ESTIMATES, M35_39_AA_ESTIMATES, M35_49_AA_ESTIMATES,
    M35_54_AA_ESTIMATES, M40_44_AA_ESTIMATES, M45_49_AA_ESTIMATES,
    M50PLUS_AA_ESTIMATES, M50_54_AA_ESTIMATES, M50_64_AA_ESTIMATES,
    M55PLUS_AA_ESTIMATES, M55_64_AA_ESTIMATES, M65_AA_ESTIMATES,
    M6_8_AA_ESTIMATES, M9_11_AA_ESTIMATES, P18_34_AA_ESTIMATES,
    P18_49_AA_ESTIMATES, P18_AA_ESTIMATES, P21_AA_ESTIMATES,
    P25_54_AA_ESTIMATES, P25_64_AA_ESTIMATES, P25_AA_ESTIMATES,
    P2_17_AA_ESTIMATES, P2_AA_ESTIMATES, P35_49_AA_ESTIMATES,
    P35_54_AA_ESTIMATES, P50_64_AA_ESTIMATES, P50_AA_ESTIMATES,
    P55_AA_ESTIMATES, P65_AA_ESTIMATES,
    CASE NETWORK_CODE
        WHEN 'LIF'  THEN F25_64_AA_ESTIMATES
        WHEN 'LMN'  THEN F25_64_AA_ESTIMATES
        WHEN 'VICE' THEN P18_49_AA_ESTIMATES
        ELSE             P25_64_AA_ESTIMATES
    END                                                                  AS DEMO_AA,
    -- [S2-SKEW] the skew demo per network, per the v6 rule: HIST reads male,
    -- the rest female. VICE defaults to female and is still unconfirmed.
    CASE NETWORK_CODE WHEN 'HIST' THEN M2_AA_ESTIMATES
                      ELSE             F2_AA_ESTIMATES END               AS SKEW_AA,
    CASE NETWORK_CODE WHEN 'HIST' THEN 'MALE' ELSE 'FEMALE' END          AS SKEW_DIRECTION,
    MEDIA_TYPE_CODE, ORIGINATOR_ID,
    SOURCE_FEED, VT AS VIEWING_TYPE,
    TRACKAGE_ID, FEED_PATTERN_IND, MARKET_BREAK_ID,
    FILE_NAME, FILE_DATE,
    CURRENT_TIMESTAMP()                                                  AS LOADED_AT
FROM ranked
QUALIFY ROW_NUMBER() OVER (
        PARTITION BY NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE,
                     PROGRAM_CODE, TELECAST_NUM
        ORDER BY FEED_RANK, WKLY_RANK, VT_RANK, TRK_RANK, MKT_RANK,
                 FILE_DATE DESC NULLS LAST, FILE_NAME DESC NULLS LAST
    ) = 1;

SELECT COUNT(*) AS ROW_COUNT,
       COUNT(DISTINCT BROADCAST_DATE) AS NIGHTS,
       SUM(IFF(MEDIAN_AGE IS NULL, 1, 0))       AS NULL_MEDIAN_AGE,
       SUM(IFF(COALESCE(P2_AA_ESTIMATES,0)=0,1,0)) AS ZERO_P2,
       SUM(IFF(COALESCE(COMMERCIAL_DURATION,0)=0,1,0)) AS ZERO_COMMERCIAL_DUR
FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT;

## 2. `CRIS_HALFHR_FACT`


In [ ]:
%%sql -r dataframe_2
-- ============================================================================
-- [S2-T1] CRIS_HALFHR_FACT      serves 2.1, 2.5, 2.6, 2.9, 2.13 and section 3
-- ----------------------------------------------------------------------------
-- Grain: NETWORK_CODE x RATING_SOURCE x BROADCAST_DATE x PROGRAM_CODE
--        x TELECAST_NUM x HALFHR_START_TIME
--
-- WHY HALF HOURS
--   Strict dayparting cannot be done at telecast grain. A 9:00-10:03p telecast
--   contributes only the part of itself that falls inside 8-11p, and a telecast
--   row cannot split itself. Every prime-norm and daypart query in this section
--   depends on that, which is why v6's 2.5 and 2.6 already read VW_HALFHR.
--
-- WHY ALL NETWORKS
--   2.9 ranks an A+E network against the cable landscape, so the table has to
--   hold the landscape. IS_AE_NETWORK lets the A+E-scoped queries filter without
--   paying for the rest. This is also what section 3 will read.
--
-- WHAT IS BAKED IN
--   The same confirmed filters and the same [S1-FEED] / [S1-VT] / [S1-DUP] /
--   [S1-DEDUP] rules as the telecast table. TOTAL_PROGRAM_IND is NOT filtered:
--   [D4] showed this warehouse stamps '00' at every grain rather than the
--   documented clock half-hour, so the filter is a no-op that only invites
--   confusion. [S1-TPI] has the detail.
--
-- WHAT IS NOT
--   No date filter beyond p_hh_years_back below, and no daypart logic. The
--   daypart boundaries live in the queries, in [S1-DP].
--
-- Build order: 2 of 3.
-- ============================================================================
-- ===== PARAMETERS =====
SET p_hh_years_back = 3;   -- [S2-WIN] how much history the competitive table holds.
                           -- 3 covers FY-to-date plus the prior year that 3.2
                           -- compares against, with a year of slack. The A+E
                           -- networks alone would fit far more; the landscape is
                           -- what makes this expensive.
SET hh_from = (SELECT DATEADD('year', -$p_hh_years_back, CURRENT_DATE()));

CREATE OR REPLACE TABLE AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
CLUSTER BY (BROADCAST_DATE, NETWORK_CODE)
AS
WITH src AS (
    SELECT
        NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE, DAY_OF_WEEK,
        PROGRAM_CODE, TELECAST_NUM,
        PROGRAM_NAME, EPISODE_NAME, CT_SEASON, IS_PREMIERE_CT,
        -- [S3-GENRE2] four genre columns, not one. CT_GENRE_1 alone cannot
        -- separate WWE SMACKDOWN from TOURNAMENT OF CHAMPIONS: both read
        -- NON-FICTION, and it is CT_GENRE_2 that says SPORTS against
        -- REALITY COMPETITION. A&E's R_1 ranker prints a Genre 2 column and
        -- excludes the sports rows; ours could not, so WWE SMACKDOWN came
        -- back at #3 in a ranker it does not appear in at all.
        --
        -- CT_COMBINED_GENRE is the concatenation A&E display, carried so the
        -- answer layer does not rebuild it.
        --
        -- PROG_TYPE_ALPHA is Nielsen's own taxonomy and is here because
        -- 3.3's p_genre_col offers it as the fallback for when Cable Tracks
        -- is thin on competitors - and that option was BROKEN, because the
        -- column existed only on CRIS_TELECAST_FACT. It is not
        -- hypothetical either: every WWE RIVALS row on COM and every ESPN
        -- row in the R_1 window carries CT_GENRE_1 NULL, which is exactly
        -- the case the fallback was written for.
        CT_GENRE_1, CT_GENRE_2, CT_GENRE_3, CT_COMBINED_GENRE,
        PROG_TYPE_ALPHA,
        START_TIME, END_TIME, EVENT_START_HR, EVENT_START_MIN,
        HALFHR_START_TIME, HH_EVENT_START_HR, HH_EVENT_START_MIN,
        CONTRIBUTING_DURATION, COMMERCIAL_DURATION,
    -- [S2-HOL] IS_HOLIDAY, computed the same way as in CRIS_TELECAST_FACT so the
    -- three tables agree. It is derived from PROGRAM_NAME, not from a Nielsen
    -- field: on Lifetime the holiday marker lives on the slate the film was
    -- bought under (MOVIE- ORIGINAL HOL PREM) while the film's own title is in
    -- EPISODE_NAME. Only films can be TRUE, because MOVIE_IND = 1 is part of the
    -- rule, so a holiday special that is not a movie is not reachable here.
    CASE WHEN MOVIE_IND = 1
           AND ( UPPER(PROGRAM_NAME) LIKE '% HOL %'
              OR UPPER(PROGRAM_NAME) LIKE '%HOLIDAY%' )
         THEN TRUE ELSE FALSE END                                        AS IS_HOLIDAY,
        REPEAT_IND, MOVIE_IND, IS_SPECIAL, PROG_PREMIERE_IND,
        MEDIAN_AGE,
    -- [S2-DEMO] P2 / M2 / F2 are not repeated here: the full AA block below
    -- already carries them, and 2.5 and 2.6 read them from there.
        -- ---- demos: every AA column the source view carries  [S2-DEMO] --------
    -- 60 columns, which is all of them. The table used to keep three, the three
    -- DEMO_AA could resolve to, which made the table the thing deciding which
    -- demo an answer came back in.
    --
    -- Nothing is composed or summed. These are the view's own columns under the
    -- view's own names, so a figure in an answer traces back to a column in the
    -- DDL with no translation step in between.
    --
    -- DEMO_AA stays as the default. It resolves per network, which is what lets
    -- one run cover several networks with each answering in its own demo.
    -- p_demo overrides it per query.
    F12_14_AA_ESTIMATES, F15_17_AA_ESTIMATES, F18_20_AA_ESTIMATES,
    F18_34_AA_ESTIMATES, F18_49_AA_ESTIMATES, F18_AA_ESTIMATES,
    F21_24_AA_ESTIMATES, F21_AA_ESTIMATES, F25_29_AA_ESTIMATES,
    F25_54_AA_ESTIMATES, F25_64_AA_ESTIMATES, F25_AA_ESTIMATES,
    F2_5_AA_ESTIMATES, F2_AA_ESTIMATES, F30_34_AA_ESTIMATES,
    F35_39_AA_ESTIMATES, F40_44_AA_ESTIMATES, F45_49_AA_ESTIMATES,
    F50_54_AA_ESTIMATES, F55_64_AA_ESTIMATES, F65_AA_ESTIMATES,
    F6_8_AA_ESTIMATES, F9_11_AA_ESTIMATES, HH_AA_ESTIMATES,
    M12_14_AA_ESTIMATES, M15_17_AA_ESTIMATES, M18_20_AA_ESTIMATES,
    M18_34_AA_ESTIMATES, M18_49_AA_ESTIMATES, M18_AA_ESTIMATES,
    M21_24_AA_ESTIMATES, M21_AA_ESTIMATES, M25_29_AA_ESTIMATES,
    M25_54_AA_ESTIMATES, M25_64_AA_ESTIMATES, M25_AA_ESTIMATES,
    M2_5_AA_ESTIMATES, M2_AA_ESTIMATES, M30_34_AA_ESTIMATES,
    M35_39_AA_ESTIMATES, M40_44_AA_ESTIMATES, M45_49_AA_ESTIMATES,
    M50_54_AA_ESTIMATES, M55_64_AA_ESTIMATES, M65_AA_ESTIMATES,
    M6_8_AA_ESTIMATES, M9_11_AA_ESTIMATES, P18_34_AA_ESTIMATES,
    P18_49_AA_ESTIMATES, P18_AA_ESTIMATES, P21_AA_ESTIMATES,
    P25_54_AA_ESTIMATES, P25_64_AA_ESTIMATES, P25_AA_ESTIMATES,
    P2_17_AA_ESTIMATES, P2_AA_ESTIMATES, P35_54_AA_ESTIMATES,
    P50_AA_ESTIMATES, P55_AA_ESTIMATES, P65_AA_ESTIMATES,
        MEDIA_TYPE_CODE, ORIGINATOR_ID,
        TRACKAGE_ID, FEED_PATTERN_IND, MARKET_BREAK_ID,
        FILE_NAME, FILE_DATE,
        IFF(UPPER(COALESCE(FILE_NAME,'')) LIKE 'UBD%','BIGDATA','PANEL')  AS SOURCE_FEED,
        NULLIF(TRIM(COALESCE(VIEWING_TYPE,'')),'')                        AS VT
    FROM AUDIENCE_DB.UBD.VW_HALFHR_PROGRAM_ESTIMATES_UNIFIED
    WHERE PROGRAM_PUT_INDICATOR = 0
      AND CONTRIBUTING_DURATION > 0
      AND BROADCAST_DATE >= $hh_from
),
ranked AS (
    SELECT src.*,
        IFF(SOURCE_FEED = 'BIGDATA', 0, 1)                                AS FEED_RANK,
        IFF(REGEXP_LIKE(UPPER(COALESCE(FILE_NAME,'')), '.*_CW.*'), 0, 1)  AS WKLY_RANK,
        IFF(VT IN ('1','3','5','B'), 1, 0)                                AS VT_RANK,
        IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(TRACKAGE_ID)),0) = 0, 0, 1) AS TRK_RANK,
        IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(MARKET_BREAK_ID)),0) = 0, 0, 1) AS MKT_RANK
    FROM src
)
SELECT
    NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE, DAY_OF_WEEK,
    PROGRAM_CODE, TELECAST_NUM,
    PROGRAM_NAME, EPISODE_NAME, CT_SEASON,
    -- [S3-GENRE2] see the src CTE above
    CT_GENRE_1, CT_GENRE_2, CT_GENRE_3, CT_COMBINED_GENRE,
    PROG_TYPE_ALPHA,
    START_TIME, END_TIME,
    HALFHR_START_TIME,
    -- [S1-2959] Nielsen clock, the only time representation carried. The half
    -- hour's own start is what places it in a daypart; the episode's start is
    -- what orders episodes inside a night.
    (MOD(MOD(HH_EVENT_START_HR, 24) + 18, 24) + 6) * 100
      + HH_EVENT_START_MIN                                       AS HH_START_2959,
    (MOD(MOD(EVENT_START_HR, 24) + 18, 24) + 6) * 100
      + EVENT_START_MIN                                          AS EP_START_2959,
    CONTRIBUTING_DURATION                                        AS HH_MIN,
    COMMERCIAL_DURATION,
    -- [CT-FIRSTRUN] episode level: catalogued in Cable Tracks = first run.
    -- Jill, 11 Sep 2026. IS_PREMIERE_CT, from the view, is the season premiere.
    IFF(CT_SEASON IS NOT NULL, TRUE, FALSE)                         AS IS_FIRST_RUN_CT,
    IS_PREMIERE_CT                                                  AS IS_PREMIERE_CT_SOURCE,
    -- [CT-SEASONPREM] IS_PREMIERE_CT is now OURS: the season premiere, the first
    -- first-run telecast of a season. Cable Tracks' own column is kept beside it
    -- as IS_PREMIERE_CT_SOURCE.
    --
    -- Measured, HIST GREATEST MACHINES season 1: the source column reads 1 on
    -- all eight episodes, so it marks first runs rather than the season opener
    -- and cannot answer "how did the season premiere do". Adarsh was going to
    -- confirm its meaning with the data team; that is still open.
    --
    -- The comparison is against a MIN over the season rather than ROW_NUMBER
    -- because QUALIFY below runs after this: duplicate copies of one telecast
    -- share the same date and telecast number, so whichever copy survives the
    -- dedup still matches the minimum. A ROW_NUMBER could land on a copy that
    -- QUALIFY then drops, leaving a season with no premiere at all.
    --
    -- Repeats carry no CT_SEASON, so they fall in a different partition and can
    -- never be picked.
    IFF(CT_SEASON IS NOT NULL
        AND TO_VARCHAR(BROADCAST_DATE, 'YYYYMMDD')
            || LPAD(TO_VARCHAR(TELECAST_NUM), 12, '0')
          = MIN(TO_VARCHAR(BROADCAST_DATE, 'YYYYMMDD')
                || LPAD(TO_VARCHAR(TELECAST_NUM), 12, '0'))
            OVER (PARTITION BY NETWORK_CODE, RATING_SOURCE,
                               PROGRAM_CODE, CT_SEASON),
        1, 0)                                                       AS IS_PREMIERE_CT,
    REPEAT_IND, MOVIE_IND, IS_SPECIAL,
    IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(PROG_PREMIERE_IND)),0) = 1, 1, 0) AS IS_SEASON_PREMIERE,
    CASE WHEN MOVIE_IND = 1 THEN 'Movie'
         WHEN UPPER(COALESCE(CT_GENRE_1,'')) = 'SPECIAL' OR IS_SPECIAL = 1 THEN 'Special'
         ELSE 'Series' END                                       AS CONTENT_TYPE,
    MEDIAN_AGE,
    -- [S2-DEMO] every AA column the source view carries, all 60. The src CTE
    -- above already selects them; this is the projection that materialises them
    -- into the table, and it previously carried six. A demo named in a query
    -- resolves against THIS list, so anything missing here is an invalid
    -- identifier at query time no matter what the CTE holds.
    F12_14_AA_ESTIMATES, F15_17_AA_ESTIMATES, F18_20_AA_ESTIMATES,
    F18_34_AA_ESTIMATES, F18_49_AA_ESTIMATES, F18_AA_ESTIMATES,
    F21_24_AA_ESTIMATES, F21_AA_ESTIMATES, F25_29_AA_ESTIMATES,
    F25_54_AA_ESTIMATES, F25_64_AA_ESTIMATES, F25_AA_ESTIMATES,
    F2_5_AA_ESTIMATES, F2_AA_ESTIMATES, F30_34_AA_ESTIMATES,
    F35_39_AA_ESTIMATES, F40_44_AA_ESTIMATES, F45_49_AA_ESTIMATES,
    F50_54_AA_ESTIMATES, F55_64_AA_ESTIMATES, F65_AA_ESTIMATES,
    F6_8_AA_ESTIMATES, F9_11_AA_ESTIMATES, HH_AA_ESTIMATES,
    M12_14_AA_ESTIMATES, M15_17_AA_ESTIMATES, M18_20_AA_ESTIMATES,
    M18_34_AA_ESTIMATES, M18_49_AA_ESTIMATES, M18_AA_ESTIMATES,
    M21_24_AA_ESTIMATES, M21_AA_ESTIMATES, M25_29_AA_ESTIMATES,
    M25_54_AA_ESTIMATES, M25_64_AA_ESTIMATES, M25_AA_ESTIMATES,
    M2_5_AA_ESTIMATES, M2_AA_ESTIMATES, M30_34_AA_ESTIMATES,
    M35_39_AA_ESTIMATES, M40_44_AA_ESTIMATES, M45_49_AA_ESTIMATES,
    M50_54_AA_ESTIMATES, M55_64_AA_ESTIMATES, M65_AA_ESTIMATES,
    M6_8_AA_ESTIMATES, M9_11_AA_ESTIMATES, P18_34_AA_ESTIMATES,
    P18_49_AA_ESTIMATES, P18_AA_ESTIMATES, P21_AA_ESTIMATES,
    P25_54_AA_ESTIMATES, P25_64_AA_ESTIMATES, P25_AA_ESTIMATES,
    P2_17_AA_ESTIMATES, P2_AA_ESTIMATES, P35_54_AA_ESTIMATES,
    P50_AA_ESTIMATES, P55_AA_ESTIMATES, P65_AA_ESTIMATES,
    CASE NETWORK_CODE
        WHEN 'LIF'  THEN F25_64_AA_ESTIMATES
        WHEN 'LMN'  THEN F25_64_AA_ESTIMATES
        WHEN 'VICE' THEN P18_49_AA_ESTIMATES
        ELSE             P25_64_AA_ESTIMATES
    END                                                          AS DEMO_AA,
    CASE NETWORK_CODE WHEN 'HIST' THEN M2_AA_ESTIMATES
                      ELSE             F2_AA_ESTIMATES END       AS SKEW_AA,
    -- ---- competitive universe -------------------------------------------
    MEDIA_TYPE_CODE, ORIGINATOR_ID,
    IFF(NETWORK_CODE IN ('AEN','HIST','LIF','LMN','FYI','VICE'), TRUE, FALSE)           AS IS_AE_NETWORK,
    SOURCE_FEED, VT AS VIEWING_TYPE,
    FILE_NAME, FILE_DATE,
    CURRENT_TIMESTAMP()                                          AS LOADED_AT
FROM ranked
QUALIFY ROW_NUMBER() OVER (
        PARTITION BY NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE,
                     PROGRAM_CODE, TELECAST_NUM, HALFHR_START_TIME
        ORDER BY FEED_RANK, WKLY_RANK, VT_RANK, TRK_RANK, MKT_RANK,
                 FILE_DATE DESC NULLS LAST, FILE_NAME DESC NULLS LAST
    ) = 1;

SELECT COUNT(*)                        AS ROW_COUNT,
       COUNT(DISTINCT NETWORK_CODE)    AS NETWORKS,
       SUM(IFF(IS_AE_NETWORK, 1, 0))   AS AE_ROWS,
       MIN(BROADCAST_DATE)             AS FROM_DATE,
       MAX(BROADCAST_DATE)             AS TO_DATE
FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT;

## 3. `CRIS_QH_FACT`


In [ ]:
%%sql -r dataframe_3
-- ============================================================================
-- [S1-T2] CRIS_QH_FACT      serves 1.2
-- ----------------------------------------------------------------------------
-- Grain: the telecast grain plus the quarter-hour slot
--        ... x PROGRAM_CODE x TELECAST_NUM x QTR_START_TIME
--
-- This is the table that justifies the effort. VW_QTRHR_PROGRAM holds ~295M rows
-- and 1.2 has to sweep a full season of them to build the norm per QH position,
-- even when the user only asked about a single night. Scoped to the six A+E
-- networks this lands in the hundreds of thousands of rows.
--
-- Same principle as the telecast table: classifications become columns, filters
-- stay in the query, REPEAT_IND is carried and never applied. Same [S1-FEED],
-- [S1-VT] and [S1-DEDUP] rules, because the quarter-hour view is built from the
-- same UNION ALL and has the same overlap.
--
-- The column is QTR_START_TIME. The check-in note calls it
-- QUARTER_HOUR_START_TIME; that name does not exist in the DDL.
--
-- Build order: 2 of 3.
-- ============================================================================
CREATE OR REPLACE TABLE AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT
CLUSTER BY (BROADCAST_DATE, NETWORK_CODE)
AS
WITH src AS (
    SELECT
        NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE, DAY_OF_WEEK,
        PROGRAM_CODE, TELECAST_NUM, EPISODE_ID,
        PROGRAM_NAME, EPISODE_NAME, CT_SEASON, IS_PREMIERE_CT,
        -- [S3-GENRE2] four genre columns, not one. CT_GENRE_1 alone cannot
        -- separate WWE SMACKDOWN from TOURNAMENT OF CHAMPIONS: both read
        -- NON-FICTION, and it is CT_GENRE_2 that says SPORTS against
        -- REALITY COMPETITION. A&E's R_1 ranker prints a Genre 2 column and
        -- excludes the sports rows; ours could not, so WWE SMACKDOWN came
        -- back at #3 in a ranker it does not appear in at all.
        --
        -- CT_COMBINED_GENRE is the concatenation A&E display, carried so the
        -- answer layer does not rebuild it.
        --
        -- PROG_TYPE_ALPHA is Nielsen's own taxonomy and is here because
        -- 3.3's p_genre_col offers it as the fallback for when Cable Tracks
        -- is thin on competitors - and that option was BROKEN, because the
        -- column existed only on CRIS_TELECAST_FACT. It is not
        -- hypothetical either: every WWE RIVALS row on COM and every ESPN
        -- row in the R_1 window carries CT_GENRE_1 NULL, which is exactly
        -- the case the fallback was written for.
        CT_GENRE_1, CT_GENRE_2, CT_GENRE_3, CT_COMBINED_GENRE,
        PROG_TYPE_ALPHA,
        START_TIME, END_TIME, EVENT_START_HR, EVENT_START_MIN,
        QTR_START_TIME, QTR_EVENT_START_HR, QTR_EVENT_START_MIN,
        CONTRIBUTING_DURATION,
        REPEAT_IND, MOVIE_IND, IS_SPECIAL, PROG_PREMIERE_IND,
        F12_14_AA_ESTIMATES, F15_17_AA_ESTIMATES, F18_20_AA_ESTIMATES,
        F18_34_AA_ESTIMATES, F18_49_AA_ESTIMATES, F18_AA_ESTIMATES,
        F21_24_AA_ESTIMATES, F21_AA_ESTIMATES, F25_29_AA_ESTIMATES,
        F25_54_AA_ESTIMATES, F25_64_AA_ESTIMATES, F25_AA_ESTIMATES,
        F2_5_AA_ESTIMATES, F2_AA_ESTIMATES, F30_34_AA_ESTIMATES,
        F35_39_AA_ESTIMATES, F40_44_AA_ESTIMATES, F45_49_AA_ESTIMATES,
        F50_54_AA_ESTIMATES, F55_64_AA_ESTIMATES, F65_AA_ESTIMATES,
        F6_8_AA_ESTIMATES, F9_11_AA_ESTIMATES, HH_AA_ESTIMATES,
        M12_14_AA_ESTIMATES, M15_17_AA_ESTIMATES, M18_20_AA_ESTIMATES,
        M18_34_AA_ESTIMATES, M18_49_AA_ESTIMATES, M18_AA_ESTIMATES,
        M21_24_AA_ESTIMATES, M21_AA_ESTIMATES, M25_29_AA_ESTIMATES,
        M25_54_AA_ESTIMATES, M25_64_AA_ESTIMATES, M25_AA_ESTIMATES,
        M2_5_AA_ESTIMATES, M2_AA_ESTIMATES, M30_34_AA_ESTIMATES,
        M35_39_AA_ESTIMATES, M40_44_AA_ESTIMATES, M45_49_AA_ESTIMATES,
        M50_54_AA_ESTIMATES, M55_64_AA_ESTIMATES, M65_AA_ESTIMATES,
        M6_8_AA_ESTIMATES, M9_11_AA_ESTIMATES, P18_34_AA_ESTIMATES,
        P18_49_AA_ESTIMATES, P18_AA_ESTIMATES, P21_AA_ESTIMATES,
        P25_54_AA_ESTIMATES, P25_64_AA_ESTIMATES, P25_AA_ESTIMATES,
        P2_17_AA_ESTIMATES, P2_AA_ESTIMATES, P35_54_AA_ESTIMATES,
        P50_AA_ESTIMATES, P55_AA_ESTIMATES, P65_AA_ESTIMATES,
        TRACKAGE_ID, FEED_PATTERN_IND, MARKET_BREAK_ID,
        FILE_NAME, FILE_DATE,
    -- ---- provenance and dedup keys -------------------------------------
    -- [S1-FEED] Which leg of the UNION ALL this row came from. The unified view
    -- caps the Big Data leg at start_date_key > 20250928 and leaves the Panel
    -- leg uncapped, so the same telecast could in principle arrive twice.
    -- [D1] and [D2] say it does not: Panel rows come under the NHI prefix and
    -- stop on 2025-09-28, Big Data rows come under UBDNHI and start on
    -- 2025-09-29, and across ten months not one telecast key carries both feeds.
    -- The cutover is clean, the 'UBD%' test is the right one, and this rank is
    -- a guard rather than a fix. It stays because the uncapped leg is still
    -- there and a Panel backfill would land straight on top of Big Data.
    IFF(UPPER(COALESCE(FILE_NAME,'')) LIKE 'UBD%','BIGDATA','PANEL')  AS SOURCE_FEED,
    -- [S1-VT] Spec position 97 pairs every stream with a without-VCR duplicate
    -- of itself: 1 against blank for Live, 3 against 2 for Live+SD, 5 against 4
    -- for Live+7, B against A for Live+3.
    -- Nielsen documents both the twin and its removal. "Big Data + Panel MIT
    -- Files and Updated Big Data + Panel MIT Files" states it outright: "To
    -- reduce the number of rows of data, we will report one viewing type in
    -- each MIT - rather than two viewing types. For example, in the production
    -- Live+SD MIT the viewing types reported are a 2 in position 97 and a 3 in
    -- position 97. In the impact MITs, only the rows with a 2 in position 97
    -- will be reported."
    -- So the twin is real in the production Panel files, Nielsen resolved it by
    -- keeping the with-VCR record, and this rank keeps the same one. [D3] found
    -- no trace of it in the warehouse, and Live arrives as '0' rather than
    -- blank, so either A&E already receives the impact files or the ETL drops
    -- the twin upstream. The rank is inert here and stays as a guard, now with
    -- a documented rule behind it rather than an inference.
    NULLIF(TRIM(COALESCE(VIEWING_TYPE,'')),'')                        AS VT
    FROM AUDIENCE_DB.UBD.VW_QTRHR_PROGRAM_ESTIMATES_UNIFIED
    -- [ALLNETS] every network, not the six. The filter was here because
    -- Sections 1 and 2 only ever asked about A&E networks, but a question about
    -- any other network then returned zero rows and no error, which reads as a
    -- quiet night rather than as out of scope. CRIS_HALFHR_FACT and
    -- CRIS_TELECAST_FACT already carried every network; these now match.
    WHERE PROGRAM_PUT_INDICATOR = 0
      -- [S1-TPI] TOTAL_PROGRAM_IND is not filtered here, but not for the reason
      -- the spec suggests. Positions 81-82 are documented as 01-48 clock
      -- half-hours at this grain, with '00' reserved for program records, which
      -- would have made v5's inherited '00' filter select nothing at all.
      -- [D4] says otherwise: in this warehouse every quarter-hour row carries
      -- '00' and nothing else, and no 'PP' appears on the program side either.
      -- The ETL does not reproduce the field's dual meaning. So the filter was
      -- harmless rather than fatal, and leaving it out is equally harmless.
      -- It stays out because a filter that matches everything is noise, and
      -- because the spec's reading is the one that would break if the ETL ever
      -- started following it. [S1-T1] keeps its copy for the same reason in
      -- reverse: there '00' is meaningful even if 'PP' is currently absent.
      AND CONTRIBUTING_DURATION > 0
),
ranked AS (
    SELECT src.*,
        -- [S1-DEDUP] preference order, most significant first
        IFF(SOURCE_FEED = 'BIGDATA', 0, 1)                            AS FEED_RANK,
        -- [S1-VT] with-VCR beats without-VCR. This reads the code and nothing
        -- else, which is the point: the twin is a property of position 97, not
        -- of the stream label, so the rule holds for every RATING_SOURCE the
        -- warehouse carries, ACM C3 and C7 included.
        IFF(VT IN ('1','3','5','B'), 1, 0)                            AS VT_RANK,
        -- [S1-DUP] The duplicates [D5] could not explain are daily versus weekly
        -- deliveries of the same telecast. [S1-D0] shows the pair directly:
        -- 51 keys carrying both UBDNHI_CDAILY and UBDNHI_CWPGMTP, plus the
        -- UBDNHI_CWACM pairs behind the ACM streams. Nothing else about them
        -- differs, which is why every other profiled column came back clean.
        -- Nielsen's weekly file is the settled delivery and the daily is the
        -- overnight preliminary, so weekly wins. Before this the FILE_NAME
        -- tiebreak was choosing between them alphabetically.
        IFF(REGEXP_LIKE(UPPER(COALESCE(FILE_NAME,'')), '.*_CW.*'), 0, 1) AS WKLY_RANK,
        -- trackage rows are sub-groupings of a program, not the telecast itself
        IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(TRACKAGE_ID)),0) = 0, 0, 1) AS TRK_RANK,
        -- 0 / null market break = Total US
        IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(MARKET_BREAK_ID)),0) = 0, 0, 1) AS MKT_RANK
    FROM src
)
SELECT
    -- ---- identity -------------------------------------------------------
    NETWORK_CODE, RATING_SOURCE,
    BROADCAST_DATE, DAY_OF_WEEK,
    PROGRAM_CODE, TELECAST_NUM, EPISODE_ID,
    PROGRAM_NAME, EPISODE_NAME, CT_SEASON,
    -- ---- episode-level schedule -----------------------------------------
    START_TIME, END_TIME,
    -- ---- quarter-hour slot ----------------------------------------------
    -- [S1-LEAN] the raw hour/minute parts and QTR_END_TIME are dropped: the
    -- first pair is what QH_START_2959 is built from, and the end is
    -- QTR_START_TIME plus QH_MIN.
    QTR_START_TIME,
    CONTRIBUTING_DURATION                                      AS QH_MIN,
    -- [S1-2959] two chronology keys, same Nielsen HHMM form as [S1-T1].
    --   QH_START_2959 orders the slots inside an episode. The old code sorted by
    --   START_TIME, which is the EPISODE start and therefore identical on every
    --   QH row of that episode, so the position came out arbitrary.
    --   EP_START_2959 orders episodes inside the night.
    (MOD(MOD(QTR_EVENT_START_HR, 24) + 18, 24) + 6) * 100
      + QTR_EVENT_START_MIN                                     AS QH_START_2959,
    (MOD(MOD(EVENT_START_HR, 24) + 18, 24) + 6) * 100
      + EVENT_START_MIN                                         AS EP_START_2959,
    -- ---- flags as columns ------------------------------------------------
    -- [CT-FIRSTRUN] episode level: catalogued in Cable Tracks = first run.
    -- Jill, 11 Sep 2026. IS_PREMIERE_CT, from the view, is the season premiere.
    IFF(CT_SEASON IS NOT NULL, TRUE, FALSE)                         AS IS_FIRST_RUN_CT,
    IS_PREMIERE_CT                                                  AS IS_PREMIERE_CT_SOURCE,
    -- [CT-SEASONPREM] IS_PREMIERE_CT is now OURS: the season premiere, the first
    -- first-run telecast of a season. Cable Tracks' own column is kept beside it
    -- as IS_PREMIERE_CT_SOURCE.
    --
    -- Measured, HIST GREATEST MACHINES season 1: the source column reads 1 on
    -- all eight episodes, so it marks first runs rather than the season opener
    -- and cannot answer "how did the season premiere do". Adarsh was going to
    -- confirm its meaning with the data team; that is still open.
    --
    -- The comparison is against a MIN over the season rather than ROW_NUMBER
    -- because QUALIFY below runs after this: duplicate copies of one telecast
    -- share the same date and telecast number, so whichever copy survives the
    -- dedup still matches the minimum. A ROW_NUMBER could land on a copy that
    -- QUALIFY then drops, leaving a season with no premiere at all.
    --
    -- Repeats carry no CT_SEASON, so they fall in a different partition and can
    -- never be picked.
    IFF(CT_SEASON IS NOT NULL
        AND TO_VARCHAR(BROADCAST_DATE, 'YYYYMMDD')
            || LPAD(TO_VARCHAR(TELECAST_NUM), 12, '0')
          = MIN(TO_VARCHAR(BROADCAST_DATE, 'YYYYMMDD')
                || LPAD(TO_VARCHAR(TELECAST_NUM), 12, '0'))
            OVER (PARTITION BY NETWORK_CODE, RATING_SOURCE,
                               PROGRAM_CODE, CT_SEASON),
        1, 0)                                                       AS IS_PREMIERE_CT,
    REPEAT_IND, MOVIE_IND, IS_SPECIAL,   -- [S1-LEAN] no daypart labels
    -- [S3-GENRE2] see the src CTE above
    CT_GENRE_1, CT_GENRE_2, CT_GENRE_3, CT_COMBINED_GENRE, PROG_TYPE_ALPHA,
    IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(PROG_PREMIERE_IND)),0) = 1, 1, 0)
                                                                AS IS_SEASON_PREMIERE,
    CASE WHEN MOVIE_IND = 1 THEN 'Movie'
         WHEN UPPER(COALESCE(CT_GENRE_1,'')) = 'SPECIAL' OR IS_SPECIAL = 1 THEN 'Special'
         ELSE 'Series' END                                      AS CONTENT_TYPE,
    -- ---- demos: every AA column the view carries  [S1-DEMO] --------------
    -- 60 columns, which is all of them. The table used to keep three
    -- (P25_64, F25_64, P18_49) because those were the three DEMO_AA could
    -- resolve to, and that made the table the thing deciding which demo an
    -- answer came back in. Four of the six queries had no demo parameter at
    -- all as a result: "how did History do last night in A18-49" was not a
    -- hard question, it was an unanswerable one without editing SQL.
    --
    -- Carrying the full set costs width and buys every demo Nielsen ships.
    -- Nothing is derived or summed here: these are the view's own columns
    -- under the view's own names, so a figure in an answer can be traced back
    -- to a column in the DDL without a translation step in between.
    --
    -- 14 of these do not exist in the quarter-hour view, so a demo can be
    -- available in 1.1 and not in 1.2. That is the file, not the build.
    --
    -- DEMO_AA stays. It is the per-network default and the reason a run across
    -- all six networks gives HIST A25-64 and LIF W25-64 from one execution.
    -- p_demo overrides it per query; left NULL it is what every validated
    -- figure was measured in.
    F12_14_AA_ESTIMATES, F15_17_AA_ESTIMATES, F18_20_AA_ESTIMATES,
    F18_34_AA_ESTIMATES, F18_49_AA_ESTIMATES, F18_AA_ESTIMATES,
    F21_24_AA_ESTIMATES, F21_AA_ESTIMATES, F25_29_AA_ESTIMATES,
    F25_54_AA_ESTIMATES, F25_64_AA_ESTIMATES, F25_AA_ESTIMATES,
    F2_5_AA_ESTIMATES, F2_AA_ESTIMATES, F30_34_AA_ESTIMATES,
    F35_39_AA_ESTIMATES, F40_44_AA_ESTIMATES, F45_49_AA_ESTIMATES,
    F50_54_AA_ESTIMATES, F55_64_AA_ESTIMATES, F65_AA_ESTIMATES,
    F6_8_AA_ESTIMATES, F9_11_AA_ESTIMATES, HH_AA_ESTIMATES,
    M12_14_AA_ESTIMATES, M15_17_AA_ESTIMATES, M18_20_AA_ESTIMATES,
    M18_34_AA_ESTIMATES, M18_49_AA_ESTIMATES, M18_AA_ESTIMATES,
    M21_24_AA_ESTIMATES, M21_AA_ESTIMATES, M25_29_AA_ESTIMATES,
    M25_54_AA_ESTIMATES, M25_64_AA_ESTIMATES, M25_AA_ESTIMATES,
    M2_5_AA_ESTIMATES, M2_AA_ESTIMATES, M30_34_AA_ESTIMATES,
    M35_39_AA_ESTIMATES, M40_44_AA_ESTIMATES, M45_49_AA_ESTIMATES,
    M50_54_AA_ESTIMATES, M55_64_AA_ESTIMATES, M65_AA_ESTIMATES,
    M6_8_AA_ESTIMATES, M9_11_AA_ESTIMATES, P18_34_AA_ESTIMATES,
    P18_49_AA_ESTIMATES, P18_AA_ESTIMATES, P21_AA_ESTIMATES,
    P25_54_AA_ESTIMATES, P25_64_AA_ESTIMATES, P25_AA_ESTIMATES,
    P2_17_AA_ESTIMATES, P2_AA_ESTIMATES, P35_54_AA_ESTIMATES,
    P50_AA_ESTIMATES, P55_AA_ESTIMATES, P65_AA_ESTIMATES,
    CASE NETWORK_CODE
        WHEN 'LIF'  THEN F25_64_AA_ESTIMATES
        WHEN 'LMN'  THEN F25_64_AA_ESTIMATES
        WHEN 'VICE' THEN P18_49_AA_ESTIMATES
        ELSE             P25_64_AA_ESTIMATES
    END                                                        AS DEMO_AA,
    -- ---- provenance ------------------------------------------------------
    SOURCE_FEED, VT AS VIEWING_TYPE,
    TRACKAGE_ID, FEED_PATTERN_IND, MARKET_BREAK_ID,
    FILE_NAME, FILE_DATE,          -- [S1-LEAN] FILE_NAME_WO_REVISION dropped
    CURRENT_TIMESTAMP()                                         AS LOADED_AT
FROM ranked
QUALIFY ROW_NUMBER() OVER (
        PARTITION BY NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE,
                     PROGRAM_CODE, TELECAST_NUM, QTR_START_TIME
        ORDER BY FEED_RANK, WKLY_RANK, VT_RANK, TRK_RANK, MKT_RANK,
                 FILE_DATE DESC NULLS LAST, FILE_NAME DESC NULLS LAST
    ) = 1;

SELECT COUNT(*)                       AS ROW_COUNT,
       COUNT(DISTINCT BROADCAST_DATE) AS NIGHTS,
       ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT BROADCAST_DATE),0), 0) AS ROWS_PER_NIGHT,
       SUM(IFF(SOURCE_FEED='BIGDATA',1,0)) AS BIGDATA_ROWS,
       SUM(IFF(SOURCE_FEED='PANEL',1,0))   AS PANEL_ROWS
FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT;

## 4. `CRIS_LATEST_DATE`


In [ ]:
%%sql -r dataframe_4
-- ============================================================================
-- [S1-T0] CRIS_LATEST_DATE
-- ----------------------------------------------------------------------------
-- Smallest of the three tables and the cheapest win. The notebook resolved the
-- latest available date 29 separate times, always by scanning a unified view
-- just to get a MAX. This replaces all of them with a lookup of a few rows.
-- This is the "pre-resolved latest-date information" named in the check-in.
--
-- Grain: NETWORK_CODE x RATING_SOURCE
--
-- Two max dates, not one. A&E confirmed Total Program and Quarter Hour are
-- delivered together and their latest dates are expected to match, so
-- QH_LAG_DAYS should read 0. It is kept precisely because it was not always 0:
-- that gap is the open investigation, and 1.2 still resolves its target night
-- from the QH side so it cannot silently return an empty result.
--
-- Build order: 3 of 3. It reads from the other two, which is why it is last.
-- ============================================================================
CREATE OR REPLACE TABLE AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE AS
SELECT
    t.NETWORK_CODE,
    t.RATING_SOURCE,
    t.LATEST_BROADCAST_DATE,
    q.LATEST_QH_BROADCAST_DATE,
    DATEDIFF('day', q.LATEST_QH_BROADCAST_DATE, t.LATEST_BROADCAST_DATE) AS QH_LAG_DAYS,
    -- last fully closed Sunday, the trailing-12m anchor used across the notebook
    DATEADD('day', -MOD(DAYOFWEEKISO(t.LATEST_BROADCAST_DATE),7), t.LATEST_BROADCAST_DATE)
                                                          AS LATEST_FULL_SUNDAY,
    t.TELECAST_ROWS,
    CURRENT_TIMESTAMP()                                   AS REFRESHED_AT
FROM (
    SELECT NETWORK_CODE, RATING_SOURCE,
           MAX(BROADCAST_DATE) AS LATEST_BROADCAST_DATE,
           COUNT(*)            AS TELECAST_ROWS
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
    GROUP BY 1,2
) t
LEFT JOIN (
    SELECT NETWORK_CODE, RATING_SOURCE,
           MAX(BROADCAST_DATE) AS LATEST_QH_BROADCAST_DATE
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT
    GROUP BY 1,2
) q ON q.NETWORK_CODE = t.NETWORK_CODE AND q.RATING_SOURCE = t.RATING_SOURCE;

SELECT * FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
ORDER BY NETWORK_CODE, RATING_SOURCE;

## 5. `CRIS_NETWORK_UNIVERSE`


In [ ]:
%%sql -r dataframe_5
-- ============================================================================
-- [S3-T0] CRIS_NETWORK_UNIVERSE      serves 3.1, 3.2, 3.3, 3.4 and 2.9
-- ----------------------------------------------------------------------------
-- Grain: NETWORK_CODE
--
-- WHY THIS EXISTS
--   The competitive universe is currently written out inside 2.9 and would be
--   written out again in all four queries here. Five copies of the same
--   eighteen-code NOT IN, the same MEDIA_TYPE_CODE test and the same 'PV'
--   artifact, and five places to edit when A&E revises the list.
--
-- THE TWO LISTS
--   They are not the same set and nobody has said which governs.
--     PRD:  eighteen network NAMES, listed in section 3.1
--     A&E:  eighteen STATION CODES, sent over Teams in Jul-2026, which v6 uses
--   Differences that matter: HLN, CNNH, ESPNN and NBATV are absent from the code
--   list and therefore stay IN the universe; NMX and ESPD are out under both.
--   Resolved with Nielsen RDN in Jun-2026: MSNBC shares station code 7801 with
--   MS NOW, so legacy MSNBC rows drop under the code list, which can move a rank
--   by one against older validations. NewsNation's 6788 was WGN America until
--   Feb-2021, so pre-2021 windows lose WGN America too.
--   Both are materialised. p_excl_by in each query picks one.
--
-- 3.4 IS THE EXCEPTION
--   Its universe INCLUDES news, sports and diginets, confirmed by A&E. Same
--   section, opposite rule to 3.1 and 3.2, which is exactly why the flags are a
--   table column rather than a filter someone has to remember.
--
-- [S3-CTCOV] CABLE TRACKS COVERAGE                                   NEW IN v19
--   CT_SEASON, CONTENT_TYPE and the two genre fields arrive through the Cable
--   Tracks join, not from the Nielsen file. Measured Aug-20-2026 against
--   VW_TOTAL_PROGRAM_ESTIMATES_UNIFIED directly, so this is a source limit and
--   not something the build dropped:
--
--       the six A&E networks   98.1%   carry CT_SEASON on premieres
--       HGTV                   97.4%
--       BRVO                   97.4%
--       FOX                    26.7%   <- and everything below it
--       CW                     25.3%
--       TNT                    21.0%
--       ... down to about 1%
--
--   So the tracked set is EIGHT networks, not the six. Nothing sits between
--   26.7% and 97.4%, which is why a threshold is safe rather than arbitrary.
--
--   The partial band is more dangerous than the empty one. A season question on
--   FOX does not return nothing; it returns a norm built from a quarter of the
--   telecasts, which looks like an answer. Zero rows at least look odd.
--
--   Measured here off CRIS_HALFHR_FACT rather than CRIS_TELECAST_FACT,
--   because this table is build order 1 of 4 and the all-net table is 3 of 4.
--   Half-hour grain weights a long programme more heavily than a short one, so
--   CT_SEASON_PCT here will not match the telecast-grain figures quoted above to
--   the decimal. It does not have to: the gap it has to resolve is 70 points
--   wide. If the two ever disagree about which side of the threshold a network
--   falls on, that disagreement is the finding.
--
--   REPEAT_IND = 0 is not optional. Cable Tracks attaches a season to PREMIERES
--   only, so every repeat carries NULL. Without it the rate collapses on every
--   network including A&E, and the result reads as no coverage anywhere.
--
--   UNCONFIRMED: why HGTV and Bravo specifically. Almost certainly a competitive
--   set A&E asked to have tracked, most likely against LIF and FYI. If A&E hold
--   a list of tracked networks, that list governs and this threshold is only a
--   proxy for it. Open with them.
--
-- [S3-NAME] THERE IS NO NETWORK NAME                                 NEW IN v19
--   Checked every source view Aug-20-2026: NETWORK_CODE, ORIGINATOR_ID,
--   ORIGINATOR_TYPE, NETWORK_OWNER, CT_NETWORK_OWNER. No display name anywhere.
--   NETWORK_OWNER exists on the source views but is the corporate parent, not
--   the channel, so it cannot resolve "Discovery" to a code either. It is NOT
--   carried here: CRIS_HALFHR_FACT, which this table is built from, does not
--   select it. Adding it would mean rebuilding the half-hour fact, which is not
--   worth it for a column that solves nothing.
--   Mapping a spoken network name to a code has to live in the planner. There is
--   nothing in the warehouse to join it to.
--
-- Build order: 1 of 4.
-- ============================================================================
CREATE OR REPLACE TABLE AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE AS
WITH seen AS (
    -- every network the half-hour fact actually carries, with the media type
    -- and station code it carries them under
    SELECT NETWORK_CODE,
           MAX(MEDIA_TYPE_CODE)                                AS MEDIA_TYPE_CODE,
           MAX(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(ORIGINATOR_ID)), -1)) AS ORIGINATOR_ID,
           COUNT(*)                                            AS HALFHRS,
           MIN(BROADCAST_DATE)                                 AS FROM_DATE,
           MAX(BROADCAST_DATE)                                 AS TO_DATE
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
    GROUP BY NETWORK_CODE
),
ct AS (
    -- [S3-CTCOV] premiere half hours, and how many of them Cable Tracks has
    -- catalogued. Trailing 12 months rather than all history: if Cable Tracks
    -- started covering a network recently, a lifetime rate hides it.
    SELECT NETWORK_CODE,
           COUNT(*)            AS PREMIERE_HALFHRS,
           COUNT(CT_SEASON)    AS WITH_SEASON
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
    WHERE REPEAT_IND     = 0
      AND RATING_SOURCE  = 'Live+SD'
      AND BROADCAST_DATE >= DATEADD('month', -12, CURRENT_DATE())
    GROUP BY NETWORK_CODE
)
SELECT
    s.NETWORK_CODE,
    s.MEDIA_TYPE_CODE,
    s.ORIGINATOR_ID,
    s.HALFHRS, s.FROM_DATE, s.TO_DATE,
    IFF(s.NETWORK_CODE IN ('AEN','HIST','LIF','LMN','FYI','VICE'), TRUE, FALSE)      AS IS_AE,
    -- 'NHI' is the cable universe; it is what keeps broadcast diginets out of
    -- 3.1 and 3.2. 3.4 wants them in, so it reads IS_BROADCAST instead.
    IFF(s.MEDIA_TYPE_CODE = 'NHI', TRUE, FALSE)                AS IS_CABLE,
    IFF(s.MEDIA_TYPE_CODE <> 'NHI', TRUE, FALSE)               AS IS_BROADCAST,
    -- 'PV' is a data artifact, found during the 3.1 validation. Out of every
    -- universe, under either list.
    IFF(UPPER(s.NETWORK_CODE) = 'PV', TRUE, FALSE)             AS IS_ARTIFACT,
    -- entertainment cable, the two readings side by side
    IFF(s.MEDIA_TYPE_CODE = 'NHI'
        AND UPPER(s.NETWORK_CODE) <> 'PV'
        AND s.ORIGINATOR_ID NOT IN (
                  6521,6202,4964,8889,6204,7287,2263,7401,7443,9042,
                  6560,7564,7801,9772,9540,10680,6788,9897),
        TRUE, FALSE)                                           AS IS_ENT_BY_CODE,
    IFF(s.MEDIA_TYPE_CODE = 'NHI'
        AND UPPER(s.NETWORK_CODE) <> 'PV'
        AND UPPER(s.NETWORK_CODE) NOT IN (
                  'CNBC','CNN','ESDN','ESNU','ESPN','ESPN2','FBN','FOXN','FS1','FS2',
                  'GOLF','MLBN','MSNW','NBAT','NFLN','NMX','NWSN','TENN'),
        TRUE, FALSE)                                           AS IS_ENT_BY_NAME,
    -- [S3-CTCOV] Cable Tracks reach, as a measured rate and as a gate
    c.PREMIERE_HALFHRS,
    ROUND(100.0 * c.WITH_SEASON / NULLIF(c.PREMIERE_HALFHRS, 0), 1)
                                                               AS CT_SEASON_PCT,
    -- 0.90 sits inside a 70 point gap, so the flag is stable against noise.
    -- Networks with too little premiere volume to measure are FALSE rather than
    -- NULL: the resolver asks one boolean, and "unknown" must gate the same way
    -- as "no", or an unmeasurable network answers a season question.
    IFF(COALESCE(c.PREMIERE_HALFHRS, 0) >= 200
        AND c.WITH_SEASON / NULLIF(c.PREMIERE_HALFHRS, 0) >= 0.90,
        TRUE, FALSE)                                           AS HAS_CT_COVERAGE,
    CURRENT_TIMESTAMP()                                        AS LOADED_AT
FROM seen s
LEFT JOIN ct c ON c.NETWORK_CODE = s.NETWORK_CODE;

-- The two lists disagreeing is the point of this check, not a fault. Any network
-- where IS_ENT_BY_CODE and IS_ENT_BY_NAME differ is one A&E has to rule on.
SELECT NETWORK_CODE, MEDIA_TYPE_CODE, ORIGINATOR_ID, HALFHRS,
       IS_ENT_BY_CODE, IS_ENT_BY_NAME
FROM AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE
WHERE IS_ENT_BY_CODE <> IS_ENT_BY_NAME
ORDER BY HALFHRS DESC;

-- [S3-CTCOV] and the coverage gate itself. Expect eight TRUE rows: the six A&E
-- networks plus HGTV and BRVO. A ninth appearing is a finding, not a bug - it
-- means A&E extended the tracking and the planner can answer more than it did.
-- Anything between 60 and 90 is the dangerous band and should be read here
-- before it is read in an answer.
SELECT NETWORK_CODE, IS_AE,
       PREMIERE_HALFHRS, CT_SEASON_PCT, HAS_CT_COVERAGE
FROM AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE
WHERE PREMIERE_HALFHRS >= 200
ORDER BY CT_SEASON_PCT DESC NULLS LAST
LIMIT 30;

## 6. `CRIS_FY_CALENDAR`


In [ ]:
%%sql -r dataframe_6
-- ============================================================================
-- [S3-T1] CRIS_FY_CALENDAR      serves 3.2, 3.3 and 2.8
-- ----------------------------------------------------------------------------
-- Grain: FISCAL_YEAR
--
-- This is now a thin view over A&E's own date dimension, not a calendar of
-- ours. It used to derive the fiscal year from a rule, the last Monday of
-- September, generated twenty years forward. The rule was right, and that is
-- exactly why it should go: a rule that happens to agree today is still a
-- second source of truth, and the day A&E moves a boundary we would not find
-- out, we would simply start disagreeing with them by a week.
--
-- Reference_DB.MDM.Dim_DATE is A&E's, and it carries FISCAL_YEAR, NIELSEN_WEEK,
-- NIELSEN_MONTH, NIELSEN_QUARTER and NIELSEN_YEAR per calendar date. Confirmed
-- against it: FY26 runs from 2025-09-29, and the Nielsen month of October 2025
-- opens on the same day, which is the boundary Kristen gave when she confirmed
-- FY25 ended 2025-09-28. Our generated table produced the same dates. The
-- switch changes no figure; it changes who owns the number.
--
-- The aligned year-ago range stays materialised here. 3.2 compares a fiscal
-- year to date against the same number of days a year earlier, and that
-- alignment is CRIS logic rather than calendar data, so it does not belong in
-- A&E's dimension.
--
-- Nielsen months, quarters and years are deliberately NOT surfaced through this
-- table. Those are resolved by the planner before a query runs, and a query that
-- can look them up is a query that can be tempted to interpret a phrase.
--
-- Build order: 2 of 4.
-- ============================================================================
CREATE OR REPLACE TABLE AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR AS
WITH bounds AS (
    SELECT TRY_TO_NUMBER(FISCAL_YEAR)  AS FISCAL_YEAR,
           MIN(CALENDAR_DATE)          AS FY_START,
           MAX(CALENDAR_DATE)          AS FY_END
    FROM Reference_DB.MDM.Dim_DATE
    WHERE FISCAL_YEAR IS NOT NULL
    GROUP BY TRY_TO_NUMBER(FISCAL_YEAR)
)
SELECT
    FISCAL_YEAR,
    FY_START,
    FY_END,
    -- the same fiscal year one year earlier, for the year-over-year comparison
    LAG(FY_START) OVER (ORDER BY FISCAL_YEAR)                  AS PRIOR_FY_START,
    LAG(FY_END)   OVER (ORDER BY FISCAL_YEAR)                  AS PRIOR_FY_END,
    CURRENT_TIMESTAMP()                                        AS LOADED_AT
FROM bounds
ORDER BY FISCAL_YEAR;

-- FY26 must read 2025-09-29 and FY25 must read 2024-09-30, the dates Kristen
-- confirmed. These now come from A&E's dimension rather than from our rule, so
-- a failure here means the dimension moved and every fiscal window in sections
-- 2 and 3 moves with it. That is worth knowing on the day it happens.
SELECT FISCAL_YEAR, FY_START, FY_END, PRIOR_FY_START, PRIOR_FY_END,
       IFF( (FISCAL_YEAR = 2026 AND FY_START = '2025-09-29')
         OR (FISCAL_YEAR = 2025 AND FY_START = '2024-09-30'),
            'OK', 'CHECK - does not match the confirmed A&E boundary')
                                                               AS ANCHOR_CHECK
FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR
WHERE FISCAL_YEAR BETWEEN 2023 AND 2027
ORDER BY FISCAL_YEAR;

## Checks

Run these after any rebuild. The first three prove the dedup held; the last one
proves the tables agree with each other about what "now" means.

In [ ]:
%%sql -r dataframe_8
/* ============================================================================
   FILE       [S1-T9] uniqueness checks
   PURPOSE    The uniqueness validation named in the check-in.
   VALIDATION Both STRICT rows read DUP_KEYS = 0. Anything else means the
              filtering or the dedup needs another look, and [S1-D0] says on
              which field the key is splitting.
   ============================================================================ */
-- The check-in states the keys as PROGRAM_CODE + TELECAST_NUM and
-- PROGRAM_CODE + TELECAST_NUM + QUARTER_HOUR_START_TIME. Taken literally those
-- repeat by construction, because the table holds six networks and several
-- rating sources, so both forms are reported:
--   STRICT  the full grain of the table, which is what must be unique
--   LITERAL the key exactly as written in the note, for the record
SELECT 'CRIS_TELECAST_FACT' AS TABLE_NAME, 'STRICT' AS KEY_FORM,
       COUNT(*) AS DUP_KEYS, COALESCE(SUM(N - 1), 0) AS EXCESS_ROWS
FROM (SELECT COUNT(*) AS N
      FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
      GROUP BY NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE,
               PROGRAM_CODE, TELECAST_NUM
      HAVING COUNT(*) > 1)
UNION ALL
SELECT 'CRIS_QH_FACT', 'STRICT',
       COUNT(*), COALESCE(SUM(N - 1), 0)
FROM (SELECT COUNT(*) AS N
      FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT
      GROUP BY NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE,
               PROGRAM_CODE, TELECAST_NUM, QTR_START_TIME
      HAVING COUNT(*) > 1)
UNION ALL
SELECT 'CRIS_TELECAST_FACT', 'LITERAL (note)',
       COUNT(*), COALESCE(SUM(N - 1), 0)
FROM (SELECT COUNT(*) AS N
      FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
      GROUP BY PROGRAM_CODE, TELECAST_NUM
      HAVING COUNT(*) > 1)
UNION ALL
SELECT 'CRIS_QH_FACT', 'LITERAL (note)',
       COUNT(*), COALESCE(SUM(N - 1), 0)
FROM (SELECT COUNT(*) AS N
      FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT
      GROUP BY PROGRAM_CODE, TELECAST_NUM, QTR_START_TIME
      HAVING COUNT(*) > 1)
-- [S1-KEY] the identity axis itself. Every norm and every join groups on
-- PROGRAM_CODE, so what matters is that one code means one series. A code
-- carrying several names is usually a restatement between revisions and is
-- harmless; a name carrying several codes is the case that would have split a
-- season in two under the old name-based grouping, and is worth a look.
UNION ALL
SELECT 'CRIS_TELECAST_FACT', 'PROGRAM_CODE -> names',
       COUNT(*), COALESCE(SUM(N - 1), 0)
FROM (SELECT COUNT(DISTINCT PROGRAM_NAME) AS N
      FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
      GROUP BY NETWORK_CODE, PROGRAM_CODE
      HAVING COUNT(DISTINCT PROGRAM_NAME) > 1)
UNION ALL
SELECT 'CRIS_TELECAST_FACT', 'PROGRAM_NAME -> codes',
       COUNT(*), COALESCE(SUM(N - 1), 0)
FROM (SELECT COUNT(DISTINCT PROGRAM_CODE) AS N
      FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
      GROUP BY NETWORK_CODE, PROGRAM_NAME
      HAVING COUNT(DISTINCT PROGRAM_CODE) > 1)
-- [S1-OA] A watch, not a gate. Per network, how often the name parse and
-- Nielsen's code disagree inside the scope 1.3 actually reads. Reported per
-- network because the aggregate is misleading: LMN carries hundreds of Lifetime
-- Originals it acquired, where the two fields correctly say different things,
-- and totalling them with LIF hides the only number that matters.
-- The LIF row is the one to watch. It sat at 11 of 996, about 1%, on the same
-- programme names that mostly code the other way. A jump means Nielsen changed
-- how it codes, and 1.3's retention norms are built on the name.
UNION ALL
SELECT 'CRIS_TELECAST_FACT',
       'MOVIE_TYPE vs ORIG_ACQUIRED_CD, ' || NETWORK_CODE,
       COUNT(CASE WHEN DECODE(ORIG_ACQUIRED_CD,'O','ORIGINAL','A','ACQUIRED')
                       <> MOVIE_TYPE THEN 1 END),
       COUNT(*)
FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
WHERE MOVIE_IND = 1
  AND MOVIE_TYPE IN ('ORIGINAL','ACQUIRED')
  AND ORIG_ACQUIRED_CD IN ('O','A')
GROUP BY NETWORK_CODE
ORDER BY TABLE_NAME, KEY_FORM;

In [ ]:
%%sql -r dataframe_9
/* ============================================================================
   FILE       [S2-T9] uniqueness checks
   PURPOSE    The half-hour key the check-in named, plus confirmation that the
              telecast table did not move when it gained columns.
   VALIDATION Both STRICT rows read DUP_KEYS = 0. TELECAST_ROWS must equal what
              [S1-T1] reported before the extension: the columns are additive,
              so a different row count means the dedup changed and section 1's
              validated numbers are no longer safe.
   ============================================================================ */
SELECT 'CRIS_HALFHR_FACT' AS TABLE_NAME, 'STRICT' AS KEY_FORM,
       COUNT(*) AS DUP_KEYS, COALESCE(SUM(N - 1), 0) AS EXCESS_ROWS
FROM (SELECT COUNT(*) AS N
      FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
      GROUP BY NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE,
               PROGRAM_CODE, TELECAST_NUM, HALFHR_START_TIME
      HAVING COUNT(*) > 1)
UNION ALL
SELECT 'CRIS_TELECAST_FACT', 'STRICT',
       COUNT(*), COALESCE(SUM(N - 1), 0)
FROM (SELECT COUNT(*) AS N
      FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
      GROUP BY NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE,
               PROGRAM_CODE, TELECAST_NUM
      HAVING COUNT(*) > 1)
UNION ALL
SELECT 'CRIS_TELECAST_FACT', 'ROW COUNT (must match [S1-T1])',
       COUNT(*), 0
FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
UNION ALL
-- [S2-DEMO] the new columns are only useful if they are populated
SELECT 'CRIS_TELECAST_FACT', 'ROWS WITHOUT MEDIAN_AGE',
       SUM(IFF(MEDIAN_AGE IS NULL, 1, 0)), COUNT(*)
FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
UNION ALL
SELECT 'CRIS_HALFHR_FACT', 'NETWORKS IN THE LANDSCAPE',
       COUNT(DISTINCT NETWORK_CODE), COUNT(DISTINCT IFF(IS_AE_NETWORK, NETWORK_CODE, NULL))
FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
ORDER BY TABLE_NAME, KEY_FORM;

-- ============================================================================
-- [S2-SYNC] how far each prepared table reaches, against the source views.
--
-- A stale prepared table does not fail, it returns nothing, and nothing reads
-- exactly like a quiet period. 3.4 is the sharpest case: it takes its date
-- window from CRIS_TELECAST_FACT and decides daypart membership from
-- CRIS_HALFHR_FACT, so a fortnight of drift between the two empties the ranker
-- with no error anywhere.
--
-- Any prepared table sitting below its source view needs rebuilding.
-- ============================================================================
SELECT 'CRIS_TELECAST_FACT'      AS OBJECT, 'prepared' AS KIND, MAX(BROADCAST_DATE) AS MAX_DATE
FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT      WHERE RATING_SOURCE = 'Live+SD'
UNION ALL SELECT 'CRIS_HALFHR_FACT', 'prepared', MAX(BROADCAST_DATE)
FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT        WHERE RATING_SOURCE = 'Live+SD'
UNION ALL SELECT 'CRIS_QH_FACT', 'prepared', MAX(BROADCAST_DATE)
FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT            WHERE RATING_SOURCE = 'Live+SD'
UNION ALL SELECT 'VW_TOTAL_PROGRAM (source)', 'view', MAX(BROADCAST_DATE)
FROM AUDIENCE_DB.UBD.VW_TOTAL_PROGRAM_ESTIMATES_UNIFIED WHERE RATING_SOURCE = 'Live+SD'
UNION ALL SELECT 'VW_HALFHR_PROGRAM (source)', 'view', MAX(BROADCAST_DATE)
FROM AUDIENCE_DB.UBD.VW_HALFHR_PROGRAM_ESTIMATES_UNIFIED WHERE RATING_SOURCE = 'Live+SD'
UNION ALL SELECT 'VW_QTRHR_PROGRAM (source)', 'view', MAX(BROADCAST_DATE)
FROM AUDIENCE_DB.UBD.VW_QTRHR_PROGRAM_ESTIMATES_UNIFIED WHERE RATING_SOURCE = 'Live+SD'
ORDER BY MAX_DATE DESC, OBJECT;

In [ ]:
%%sql -r dataframe_10
/* ============================================================================
   FILE       [S3-T9] uniqueness and universe checks
   PURPOSE    The grain check on the new table, and the two facts about the
              universe that 3.1 through 3.4 rest on.
   VALIDATION DUP_KEYS = 0 on the STRICT key.
              BROADCAST NETWORKS must be greater than zero, or 3.4 cannot answer
              a question about TV and can only answer about cable.
              ENT CABLE DISAGREEMENTS is the count of networks the two exclusion
              lists rule differently on. It is not an error; it is the size of
              the question A&E has to answer.
   ============================================================================ */
SELECT 'CRIS_TELECAST_FACT' AS TABLE_NAME, 'STRICT' AS CHECK_,
       COUNT(*) AS N, 0 AS DETAIL
FROM (SELECT COUNT(*) AS C
      FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
      GROUP BY NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE,
               PROGRAM_CODE, TELECAST_NUM
      HAVING COUNT(*) > 1)
UNION ALL
SELECT 'CRIS_NETWORK_UNIVERSE', 'NETWORKS TOTAL', COUNT(*), 0
FROM AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE
UNION ALL
SELECT 'CRIS_NETWORK_UNIVERSE', 'CABLE NETWORKS',
       SUM(IFF(IS_CABLE, 1, 0)), SUM(IFF(IS_BROADCAST, 1, 0))
FROM AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE
UNION ALL
SELECT 'CRIS_NETWORK_UNIVERSE', 'ENT CABLE BY CODE / BY NAME',
       SUM(IFF(IS_ENT_BY_CODE, 1, 0)), SUM(IFF(IS_ENT_BY_NAME, 1, 0))
FROM AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE
UNION ALL
SELECT 'CRIS_NETWORK_UNIVERSE', 'ENT CABLE DISAGREEMENTS',
       SUM(IFF(IS_ENT_BY_CODE <> IS_ENT_BY_NAME, 1, 0)), 0
FROM AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE
UNION ALL
SELECT 'CRIS_FY_CALENDAR', 'FY26 ANCHOR = 2025-09-29',
       COUNT(*), 0
FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR
WHERE FISCAL_YEAR = 2026 AND FY_START = '2025-09-29'
ORDER BY TABLE_NAME, CHECK_;

## Check: Nielsen against Cable Tracks on premiere


In [ ]:
%%sql
SELECT
    NETWORK_CODE,
    IFF(BCAST_START_2959 >= 2000 AND BCAST_START_2959 < 2400, '8p-12a', 'other') AS DAYPART,
    COUNT(*)                                                   AS TELECASTS,
    COUNT_IF(REPEAT_IND = 0 AND IS_FIRST_RUN_CT)               AS AGREE_FIRST_RUN,
    COUNT_IF(REPEAT_IND = 1 AND NOT IS_FIRST_RUN_CT)           AS AGREE_REPEAT,
    COUNT_IF(REPEAT_IND = 0 AND NOT IS_FIRST_RUN_CT)           AS NIELSEN_ONLY,
    COUNT_IF(REPEAT_IND = 1 AND IS_FIRST_RUN_CT)               AS CT_ONLY,
    SUM(IS_PREMIERE_CT)                                        AS CT_SEASON_PREMIERES,
    SUM(IS_SEASON_PREMIERE)                                    AS NIELSEN_SEASON_PREMIERES
FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
WHERE RATING_SOURCE = 'Live+SD'
  AND NETWORK_CODE IN ('AEN','HIST','LIF','LMN','FYI','VICE')
GROUP BY NETWORK_CODE, DAYPART
ORDER BY NETWORK_CODE, DAYPART;
